# Observatorio NOA · TopicPanorama · v0.2
**Respaldo documental, balance de fuentes y concentración relativa**

Esta versión conserva la exploración de las etapas inicial y refinada e incorpora tres herramientas: comparación simultánea de ponderaciones, aporte y exclusión temporal de documentos, y concentración temática provincial.

Se puede reutilizar el ZIP de resultados de la v0.1 que ya ejecutaste o reconstruir el visor a partir de `outputs_1821` y `observatorio_noa_rag_v302`. Los análisis utilizan los perfiles preservados y funcionan en CPU.

Las propuestas de etiquetas son interpretaciones pendientes de revisión, acompañadas por pasajes. El investigador puede validarlas registrando su fundamento y evidencia.

## 1. Elegir la entrada
El modo **resultados** solicita un único ZIP o JSON de resultados v0.1/v0.2. El modo **originales** permite cargar ambas exportaciones iniciales o indicar carpetas extraídas. Dejá las rutas vacías para seleccionar archivos desde Colab.

In [1]:
import os
from pathlib import Path

MODO_CARGA = os.environ.get("NOA_MODE", "resultados")  # "resultados" u "originales"
RUTA_RESULTADOS = os.environ.get("NOA_RESULTS", "")
RUTA_INICIAL = os.environ.get("NOA_INITIAL", "")
RUTA_REFINADA = os.environ.get("NOA_REFINED", "")
CARPETA_SALIDA = Path(os.environ.get("NOA_OUTPUT", "observatorio_noa_panorama_v02_salida"))
MOSTRAR_VISOR = os.environ.get("NOA_HEADLESS", "0") != "1"
DESCARGAR_ZIP_EN_COLAB = True
if MODO_CARGA not in ["resultados", "originales"]:
    raise ValueError("Elegí MODO_CARGA = 'resultados' u 'originales'.")

## 2. Preparar las dependencias de lectura
Se instalan solo las dependencias que falten. Pyarrow se necesita únicamente para reconstruir desde las exportaciones originales. La entrada de resultados usa JSON y no necesita cargar matrices ni descargar modelos.

In [2]:
import importlib.util
import subprocess
import sys

necesarios = ["pandas", "numpy"] + (["pyarrow"] if MODO_CARGA == "originales" else [])
faltantes = [n for n in necesarios if importlib.util.find_spec(n) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *faltantes])
print("Modo de carga:", MODO_CARGA, "· ejecución en CPU")

Modo de carga: resultados · ejecución en CPU


## 3. Escribir los módulos reproducibles
El módulo analítico calcula promedios, contribuciones, exclusiones y concentraciones. El importador mantiene los controles de integridad de la v0.1. Las referencias documentales y las observaciones de catálogo quedan separadas de los datos originales.

In [3]:
%%writefile noa_analysis.py
"""Análisis descriptivos sobre perfiles preservados, sin reajustar modelos."""
from __future__ import annotations
import copy
import hashlib
import json
import math
import unicodedata
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

VERSION = "0.2.0"
PROVINCES = ["Catamarca", "La Rioja", "Santiago del Estero"]
PROPOSALS = [
    (0, "Proyectos y ejecución de actividades", "Incluye iniciativas productivas y otras actividades organizadas mediante proyectos.", ["DOC_009_CH_0026", "DOC_003_CH_0004"]),
    (1, "Competencias y articulación institucional", "Agrupa referencias a autoridades, jurisdicciones, municipios y adhesiones.", ["DOC_013_CH_0004", "DOC_014_CH_0003"]),
    (2, "Promoción y financiamiento", "Reúne fondos, microcrédito y mecanismos de apoyo financiero.", ["DOC_012_CH_0007", "DOC_009_CH_0153"]),
    (3, "Programas y comunicación de gestión", "Vincula programas con piezas y actividades de comunicación gubernamental.", ["DOC_007_CH_0003", "DOC_007_CH_0004", "DOC_011_CH_0002"]),
    (4, "Tareas y gestión de información", "Incluye tareas, relevamientos, bases de datos y etapas de trabajo; puede reflejar el género del informe.", ["DOC_002_CH_0005", "DOC_007_CH_0017"]),
    (5, "Plazos, obligaciones y sanciones", "Incluye plazos financieros y obligaciones normativas; requiere revisar sus usos heterogéneos.", ["DOC_009_CH_0193", "DOC_015_CH_0029"]),
    (6, "Adhesión y condiciones de beneficios", "Reúne beneficios, compromisos y condiciones de acceso o mantenimiento.", ["DOC_018_CH_0007", "DOC_016_CH_0003"]),
    (7, "Permisos y regulación de actividades", "Incluye permisos y operadores, junto con otros pasajes cuya afinidad necesita revisión contextual.", ["DOC_015_CH_0019", "DOC_019_CH_0002"]),
]


def normalize(value):
    return "".join(c for c in unicodedata.normalize("NFKD", str(value or "")) if not unicodedata.combining(c)).lower().replace("_", " ")


def vector(doc, layer):
    values = list(doc.get("profiles", {}).get(layer, {}).values())
    if not values or any(v is None or not math.isfinite(float(v)) for v in values):
        return None
    a = np.asarray(values, dtype=float)
    return a / a.sum() if a.sum() > 0 else None


def matches(doc, payload, context=None, ignore_province=False, ignore_exclusions=False):
    c = context or {}
    if not doc["full"] or doc["regional"] or not doc["compare"]:
        return False
    if not ignore_province and c.get("province", "all") not in ("all", doc["province"]):
        return False
    if c.get("source", "all") not in ("all", doc["source"]):
        return False
    if c.get("period", "all") not in ("all", doc["period"]):
        return False
    if c.get("universe") == "common":
        common = {r["uid"] for r in payload["coverage"] if r["initial_fulltext"] and r["refined_fulltext"]}
        if doc["uid"] not in common:
            return False
    if not ignore_exclusions and doc["uid"] in c.get("excluded", []):
        return False
    if c.get("query") and normalize(c["query"]) not in normalize(doc["title"] + " " + doc["summary"] + " " + payload["texts"].get(doc["text"], "")):
        return False
    return True


def selected_docs(payload, stage_id, layer="bt", context=None, ignore_province=False, ignore_exclusions=False):
    stage = next(s for s in payload["stages"] if s["id"] == stage_id)
    return [d for d in stage["documents"] if matches(d, payload, context, ignore_province, ignore_exclusions) and vector(d, layer) is not None]


def weights(docs, mode="documents"):
    if not docs:
        return None
    if mode == "documents":
        return np.full(len(docs), 1 / len(docs))
    counts = {s: sum(d["source"] == s for d in docs) for s in ["CFI", "SAIJ"]}
    if not all(counts.values()):
        return None
    return np.asarray([.5 / counts[d["source"]] for d in docs], dtype=float)


def mean_profile(docs, layer="bt", mode="documents"):
    w = weights(docs, mode)
    return np.sum(np.asarray([vector(d, layer) for d in docs]) * w[:, None], axis=0) if w is not None else None


def provincial_profiles(payload, stage_id, layer="bt", context=None, ignore_province=False):
    ds = selected_docs(payload, stage_id, layer, context, ignore_province)
    rows = []
    for province in PROVINCES:
        group = [d for d in ds if d["province"] == province]
        observed, balanced = mean_profile(group, layer), mean_profile(group, layer, "sources")
        rows.append({"province": province, "docs": group, "n": len(group), "cfi": sum(d["source"] == "CFI" for d in group), "saij": sum(d["source"] == "SAIJ" for d in group), "observed": observed, "balanced": balanced})
    return rows


def influence(payload, stage_id, province, topic, layer="bt", mode="documents", context=None):
    c = dict(context or {}, province=province)
    docs = selected_docs(payload, stage_id, layer, c)
    mean = mean_profile(docs, layer, mode)
    if mean is None or topic >= len(mean):
        return []
    w = weights(docs, mode)
    topic_mass = float(mean[topic])
    rows = []
    for i, doc in enumerate(docs):
        remaining = [d for d in docs if d["uid"] != doc["uid"]]
        without = mean_profile(remaining, layer, mode)
        contribution = float(w[i] * vector(doc, layer)[topic] / topic_mass) if topic_mass else None
        rows.append({"stage": stage_id, "layer": layer, "province": province, "topic": topic, "mode": mode, "doc_id": doc["id"], "uid": doc["uid"], "title": doc["title"], "source": doc["source"], "n_active": len(docs), "document_weight": float(w[i]), "document_affinity": float(vector(doc, layer)[topic]), "contribution_share": contribution, "mean_current": topic_mass, "mean_without_document": float(without[topic]) if without is not None else None, "delta_pp": float(100 * (without[topic] - topic_mass)) if without is not None else None, "leader_before": int(np.argmax(mean)), "leader_after": int(np.argmax(without)) if without is not None else None, "balance_lost": bool(mode == "sources" and remaining and without is None), "profiles_frozen": True})
    return rows


def concentration(payload, stage_id, layer="bt", mode="documents", context=None):
    groups = provincial_profiles(payload, stage_id, layer, context, ignore_province=True)
    key = "balanced" if mode == "sources" else "observed"
    available = [g for g in groups if g[key] is not None]
    if not available:
        return []
    reference = np.mean([g[key] for g in available], axis=0)
    rows = []
    for group in groups:
        p = group[key]
        for topic, r in enumerate(reference):
            docs = group["docs"]
            support = [d for d in docs if vector(d, layer)[topic] >= np.sort(vector(d, layer))[-min(2, len(reference))] - 1e-9] if p is not None else []
            rows.append({"stage": stage_id, "layer": layer, "mode": mode, "province": group["province"], "topic": topic, "n_documents": group["n"], "n_cfi": group["cfi"], "n_saij": group["saij"], "affinity": float(p[topic]) if p is not None else None, "reference": float(r), "reference_provinces": len(available), "relative_concentration": float(p[topic] / r) if p is not None and r > 0 and len(available) >= 2 else None, "support_top2": len(support) if p is not None else None, "support_doc_ids": "|".join(d["id"] for d in support), "support_cfi": sum(d["source"] == "CFI" for d in support) if p is not None else None, "support_saij": sum(d["source"] == "SAIJ" for d in support) if p is not None else None, "reference_ignores_province_filter": True})
    return rows


def enrich_payload(payload):
    payload = copy.deepcopy(payload)
    payload["schema"] = "observatorio-noa-panorama/0.2"
    payload["version"] = VERSION
    payload["viewer_generated_at"] = datetime.now(timezone.utc).isoformat()
    refined = next(s for s in payload["stages"] if s["id"] == "refined")
    chunk_ids = {c["id"] for c in refined["chunks"]}
    payload["label_proposals"] = [{"stage": "refined", "layer": "bt", "topic": topic, "label": label, "description": description, "status": "proposed", "evidence_refs": [r for r in refs if r in chunk_ids], "review_note": "Propuesta interpretativa para revisar; los pasajes ilustran usos y no certifican homogeneidad del tópico."} for topic, label, description, refs in PROPOSALS]
    payload["catalogue_reviews"] = [{"stage": "refined", "doc_id": "DOC_016", "field": "title", "status": "pending", "observation": "El título catalogado menciona bosques nativos, mientras el artículo recuperado refiere a Inversiones para Bosques Cultivados. Verificar la denominación conservando el dato original.", "evidence_refs": ["DOC_016_CH_0002"]}]
    payload["analysis_methods"] = {"profiles": "Cada perfil documental se normaliza sobre los tópicos exportados y permanece fijo al excluir documentos.", "observed": "Media con igual peso por documento dentro de la provincia.", "balanced": "Media de las medias CFI y SAIJ, con peso 0.5 por fuente. No disponible cuando falta una fuente.", "contribution": "q(d,t) = w(d) p(d,t) / suma_d w(d) p(d,t). Mide aporte al promedio de afinidad, no presupuesto ni impacto.", "leave_one_out": "Se retira un documento y se recalcula el promedio con perfiles fijos. Si se pierde una fuente, el perfil equilibrado queda no disponible.", "relative_concentration": "L(p,t) = afinidad(p,t) / media de afinidad de las provincias disponibles, cada provincia con igual peso. El filtro provincial no modifica la referencia.", "support_top2": "Número de documentos en los que el tópico se encuentra entre las dos mayores afinidades; incluye empates. En K=2 todos los documentos cumplen este criterio.", "scenario": "Las exclusiones se aplican a ambas etapas por identidad documental. Las auditorías conservan los registros originales.", "interpretation": "Las etiquetas propuestas requieren revisión. La validación experta exige una etiqueta, fundamento y referencia a evidencia exportada."}
    return payload


def load_saved_results(path):
    path = Path(path)
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if n.rsplit("/", 1)[-1] == "panorama_datos.json"]
            if len(names) != 1 or z.getinfo(names[0]).file_size > 100 * 1024 * 1024:
                raise ValueError("Se requiere un único panorama_datos.json de tamaño compatible")
            payload = json.loads(z.read(names[0]))
    else:
        payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("schema") not in ["observatorio-noa-panorama/0.1", "observatorio-noa-panorama/0.2"]:
        raise ValueError("El esquema de resultados no es compatible")
    if {s["id"] for s in payload["stages"]} != {"initial", "refined"}:
        raise ValueError("El paquete debe contener ambas etapas")
    for stage in payload["stages"]:
        dids = {d["id"] for d in stage["documents"]}
        if len(dids) != len(stage["documents"]) or len({d["uid"] for d in stage["documents"]}) != len(dids):
            raise ValueError("Identidad documental duplicada")
        for chunk in stage["chunks"]:
            if chunk["doc"] not in dids or chunk["text"] not in payload["texts"]:
                raise ValueError("Referencia de documento o texto incompleta")
            if hashlib.sha256(payload["texts"][chunk["text"]].encode()).hexdigest() != chunk["text"]:
                raise ValueError("La huella de un texto no coincide con su contenido")
        for doc in stage["documents"]:
            for values in doc["profiles"].values():
                if any(v is not None and (not math.isfinite(float(v)) or float(v) < 0) for v in values.values()):
                    raise ValueError("Perfil documental inválido")
    payload = enrich_payload(payload)
    payload["reused_results"] = {"filename": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "embedding_checks": "preservados del paquete; las matrices no se vuelven a cargar por esta ruta"}
    return payload


def export_analyses(payload, directory):
    directory = Path(directory)
    profile_rows, influence_rows, lift_rows = [], [], []
    for stage in payload["stages"]:
        for layer in stage["layers"]:
            for group in provincial_profiles(payload, stage["id"], layer):
                for mode, key in [("documents", "observed"), ("sources", "balanced")]:
                    p = group[key]
                    for topic in range(len(stage["layers"][layer]["topics"])):
                        profile_rows.append({"stage": stage["id"], "layer": layer, "province": group["province"], "mode": mode, "topic": topic, "affinity": float(p[topic]) if p is not None else None, "n_documents": group["n"], "n_cfi": group["cfi"], "n_saij": group["saij"], "available": p is not None})
                        influence_rows += influence(payload, stage["id"], group["province"], topic, layer, mode)
            for mode in ["documents", "sources"]:
                lift_rows += concentration(payload, stage["id"], layer, mode)
    tables = {"perfiles_observados_y_equilibrados.csv": profile_rows, "aporte_documental_y_exclusion.csv": influence_rows, "concentracion_relativa.csv": lift_rows, "etiquetas_propuestas.csv": payload["label_proposals"], "revision_catalogacion.csv": payload["catalogue_reviews"]}
    for name, rows in tables.items():
        pd.DataFrame(rows).to_csv(directory / name, index=False, encoding="utf-8-sig")
    (directory / "metodos_analiticos.json").write_text(json.dumps(payload["analysis_methods"], ensure_ascii=False, indent=2), encoding="utf-8")
    return {name: len(rows) for name, rows in tables.items()}

Writing noa_analysis.py


In [4]:
%%writefile noa_panorama.py
"""Visor de resultados existentes. No ajusta LDA, BERTopic ni embeddings."""
from __future__ import annotations

import ast
import ctypes as C
import ctypes.util
import hashlib
import json
import math
import re
import unicodedata
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

from noa_analysis import enrich_payload, export_analyses

VERSION = "0.2.0"
PROVINCES = ["Catamarca", "La Rioja", "Santiago del Estero"]


def normalized(value):
    value = unicodedata.normalize("NFKD", str(value or ""))
    return re.sub(r"[^a-z0-9]+", " ", "".join(c for c in value if not unicodedata.combining(c)).lower()).strip()


def text(value):
    if value is None or (not isinstance(value, (list, dict)) and pd.isna(value)):
        return ""
    return str(value)


def number(value):
    try:
        n = float(value)
        return round(n, 9) if math.isfinite(n) else None
    except (ValueError, TypeError):
        return None


def truth(value):
    if isinstance(value, str):
        return value.lower() in ("true", "1", "yes", "si", "sí")
    return bool(value) if pd.notna(value) else False


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def safe_target(root, name):
    name = name.replace("\\", "/")
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or re.match(r"^[A-Za-z]:", name):
        raise ValueError(f"Ruta no permitida dentro del archivo: {name}")
    target = root.joinpath(*p.parts)
    if not target.resolve().is_relative_to(root.resolve()):
        raise ValueError("La ruta sale del directorio de extracción")
    return target


def extract_rar(source, root):
    """Libarchive del sistema; compatible con el RAR5 suministrado y con Colab/Linux."""
    library = ctypes.util.find_library("archive")
    if not library:
        raise RuntimeError("RAR requiere libarchive. En este entorno, extraé el RAR y usá la carpeta o comprimila como ZIP.")
    lib = C.CDLL(library)
    def bind(name, result, arguments):
        f = getattr(lib, name)
        f.restype, f.argtypes = result, arguments
        return f
    new = bind("archive_read_new", C.c_void_p, [])
    filters = bind("archive_read_support_filter_all", C.c_int, [C.c_void_p])
    formats = bind("archive_read_support_format_all", C.c_int, [C.c_void_p])
    opening = bind("archive_read_open_filename", C.c_int, [C.c_void_p, C.c_char_p, C.c_size_t])
    next_header = bind("archive_read_next_header", C.c_int, [C.c_void_p, C.POINTER(C.c_void_p)])
    pathname = bind("archive_entry_pathname", C.c_char_p, [C.c_void_p])
    size = bind("archive_entry_size", C.c_int64, [C.c_void_p])
    kind = bind("archive_entry_filetype", C.c_uint, [C.c_void_p])
    read = bind("archive_read_data", C.c_ssize_t, [C.c_void_p, C.c_void_p, C.c_size_t])
    error = bind("archive_error_string", C.c_char_p, [C.c_void_p])
    free = bind("archive_read_free", C.c_int, [C.c_void_p])
    a = new()
    filters(a); formats(a)
    try:
        if opening(a, str(source).encode(), 10240) < 0:
            raise RuntimeError(text(error(a)))
        total = 0
        while True:
            entry = C.c_void_p()
            status = next_header(a, C.byref(entry))
            if status == 1:
                break
            if status < 0:
                raise RuntimeError(error(a).decode(errors="replace"))
            target = safe_target(root, pathname(entry).decode("utf-8", errors="strict"))
            if kind(entry) == 0o040000:
                target.mkdir(parents=True, exist_ok=True)
                continue
            if kind(entry) != 0o100000:
                raise ValueError("Solo se admiten archivos regulares y carpetas")
            nbytes = size(entry)
            total += nbytes
            if total > 512 * 1024 * 1024:
                raise ValueError("La exportación supera el límite de extracción de 512 MiB")
            target.parent.mkdir(parents=True, exist_ok=True)
            buffer = C.create_string_buffer(1024 * 1024)
            written = 0
            with target.open("wb") as out:
                while True:
                    n = read(a, buffer, len(buffer))
                    if n == 0:
                        break
                    if n < 0:
                        raise RuntimeError(error(a).decode(errors="replace"))
                    out.write(buffer.raw[:n]); written += n
            if written != nbytes:
                raise ValueError(f"Tamaño extraído incorrecto: {target.name}")
    finally:
        free(a)


def open_export(source, workdir):
    source = Path(source).expanduser().resolve()
    if source.is_dir():
        h = hashlib.sha256()
        for p in sorted(source.rglob("*")):
            if p.is_file() and not any(part.startswith(".") for part in p.relative_to(source).parts):
                h.update(p.relative_to(source).as_posix().encode())
                h.update(b"\x00")
                h.update(sha256(p).encode())
                h.update(b"\x00")
        return source, {"filename": source.name, "sha256": h.hexdigest(), "kind": "directory", "hash_basis": "rutas relativas y SHA-256 de los archivos"}
    if not source.is_file():
        raise FileNotFoundError(f"No se encontró {source}")
    digest = sha256(source)
    root = Path(workdir) / digest[:16]
    marker = root / ".extraction_complete"
    if not marker.exists():
        root.mkdir(parents=True, exist_ok=True)
        if source.suffix.lower() == ".zip":
            with zipfile.ZipFile(source) as z:
                if sum(i.file_size for i in z.infolist()) > 512 * 1024 * 1024:
                    raise ValueError("El ZIP supera 512 MiB al extraerse")
                for item in z.infolist():
                    target = safe_target(root, item.filename)
                    if ((item.external_attr >> 16) & 0o170000) == 0o120000:
                        raise ValueError("No se admiten enlaces simbólicos")
                    if item.is_dir():
                        target.mkdir(parents=True, exist_ok=True)
                    else:
                        target.parent.mkdir(parents=True, exist_ok=True)
                        target.write_bytes(z.read(item))
        elif source.suffix.lower() == ".rar":
            extract_rar(source, root)
        else:
            raise ValueError("Usá una carpeta extraída, un ZIP o un RAR")
        marker.write_text(digest)
    return root, {"filename": source.name, "sha256": digest, "kind": source.suffix.lower()[1:]}


class Export:
    def __init__(self, root):
        self.root = Path(root)

    def file(self, name, required=True):
        matches = list(self.root.rglob(name))
        if len(matches) != 1:
            if not matches and not required:
                return None
            raise ValueError(f"Se esperaba un único {name}; encontrados: {len(matches)}")
        return matches[0]

    def table(self, name, required=True):
        p = self.file(name, required)
        if p is None:
            return pd.DataFrame()
        return pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)

    def json(self, name):
        return json.loads(self.file(name).read_text(encoding="utf-8-sig"))


def descriptor_words(descriptor):
    """Extrae la lista explícita de palabras; evita tomar ejemplos como descriptores."""
    descriptor = text(descriptor)
    match = re.search(r"\[[^\]]+\]", descriptor)
    if match:
        try:
            values = ast.literal_eval(match.group())
            if isinstance(values, list):
                return [str(v) for v in values]
        except (ValueError, SyntaxError):
            pass
    return list(dict.fromkeys(v.strip() for v in descriptor.split("|") if v.strip()))[:15]


def document_identity(row):
    year = number(row.get("year_matrix"))
    raw = "|".join([normalized(row.get("source")), normalized(row.get("province")), str(int(year)) if year else "", normalized(row.get("title"))])
    return hashlib.sha256(raw.encode()).hexdigest()[:20]


def as_topics(frame, columns):
    return {str(i): number(frame.get(c)) for i, c in enumerate(columns)}


def load_stage(root, stage_id, source, store):
    export = Export(root)
    initial = stage_id == "initial"
    manifest = export.json("run_manifest.json" if initial else "rag_run_manifest_v302.json")
    docs = export.table("documents.parquet" if initial else "rag_document_index.parquet")
    chunks = export.table("chunks_raw.parquet" if initial else "rag_chunk_index.parquet")
    for frame, label, key in [(docs, "documentos", "doc_id"), (chunks, "fragmentos", "chunk_id")]:
        if frame[key].isna().any() or frame[key].duplicated().any():
            raise ValueError(f"Identificadores ausentes o duplicados en {label} de {stage_id}")
    if not chunks.doc_id.isin(docs.doc_id).all():
        raise ValueError("Hay fragmentos sin documento de origen")
    bt_columns = [f"bt_topic_{i}" for i in range(3)] if initial else [f"sub_bt_topic_{i}" for i in range(8)]
    layers = {}
    if initial:
        ld = export.table("lda_document_topics.parquet").set_index("doc_id")
        bt = export.table("bertopic_document_topics.csv").set_index("doc_id")
        if manifest.get("selected_lda_k") != 3 or manifest.get("bertopic_topics") != 3:
            raise ValueError("El adaptador inicial v0.1 corresponde a LDA K=3 y BERTopic global de 3 tópicos. Esta exportación requiere otro adaptador.")
        for frame, columns in [(ld, [f"lda_topic_{i}" for i in range(3)]), (bt, bt_columns + ["bt_outlier"])]:
            if not set(columns).issubset(frame.columns):
                raise ValueError("La exportación inicial no incluye todos los perfiles esperados")
        info = export.table("bertopic_topic_info.csv")
        topics = []
        for _, r in info.loc[info.Topic.ge(0)].sort_values("Topic").iterrows():
            words = ast.literal_eval(r.Representation)
            topics.append({"id": str(int(r.Topic)), "words": [{"term": w, "weight": None} for w in words], "training_count": int(r.Count)})
        terms = export.table("lda_topic_terms.csv")
        lda_topics = [{"id": str(int(i)), "words": [{"term": str(r.term), "weight": number(r.weight)} for _, r in group.sort_values("rank").iterrows()]} for i, group in terms.groupby("lda_topic", sort=True)]
        layers["bt"] = {"name": "BERTopic global · 3 tópicos", "topics": topics, "basis": "Afinidades documentales exportadas, condicionadas a masa temática", "weighted_terms": False}
        layers["lda3"] = {"name": "LDA · K=3", "topics": lda_topics, "basis": "Distribuciones documentales LDA exportadas", "weighted_terms": True}
        training = export.table("bertopic_training_assignments.csv").set_index("chunk_id")
        if not training.index.isin(chunks.chunk_id).all():
            raise ValueError("Hay asignaciones de entrenamiento sin fragmento")
        policy = export.table("chunks_policy_view.parquet").set_index("chunk_id")
        crosswalk = export.table("lda_bertopic_crosswalk.csv").to_dict("records")
        robustness = export.table("bertopic_semantic_robustness.csv").to_dict("records")
    else:
        ld = bt = docs.set_index("doc_id")
        expected = bt_columns + [f"lda_k{k}_topic_{i}" for k in [2, 3] for i in range(k)]
        if not set(expected).issubset(docs.columns):
            raise ValueError("El adaptador refinado v0.1 requiere los perfiles LDA K=2/K=3 y los 8 microtópicos sustantivos de RAG v3.0.2")
        topics = []
        for _, r in export.table("rag_microtopic_descriptors.csv").sort_values("topic_id").iterrows():
            topics.append({"id": str(int(r.topic_id)), "words": [{"term": w, "weight": None} for w in descriptor_words(r.descriptor)]})
        layers["bt"] = {"name": "BERTopic sustantivo · 8 microtópicos", "topics": topics, "basis": "Afinidades documentales del nivel sustantivo exportadas", "weighted_terms": False}
        for k in [2, 3]:
            lda_topics = []
            for _, r in export.table(f"rag_lda_k{k}_descriptors.csv").sort_values("topic_id").iterrows():
                lda_topics.append({"id": str(int(r.topic_id)), "words": [{"term": w, "weight": None} for w in descriptor_words(r.descriptor)]})
            layers[f"lda{k}"] = {"name": f"LDA · K={k}" + (" · principal" if k == 2 else " · refinamiento"), "topics": lda_topics, "basis": "Distribuciones documentales LDA exportadas", "weighted_terms": False}
        crosswalk, robustness = [], []
        training = policy = None

    records, doc_by_id = [], {}
    for _, row in docs.iterrows():
        doc_id = str(row.doc_id)
        origin = text(row.get("policy_view_origin"))
        full = origin in ("full_text", "ocr_text") or (number(row.get("word_count")) or 0) > 0
        d = {"id": doc_id, "uid": document_identity(row), "title": text(row.get("title")), "province": text(row.get("province")), "source": text(row.get("source")), "year": number(row.get("year_matrix")), "year_catalog": number(row.get("year_catalog")), "period": text(row.get("period")), "link": text(row.get("link")), "summary": text(row.get("summary")), "origin": origin, "full": full, "regional": truth(row.get("regional_context")), "compare": truth(row.get("include_in_province_comparison")), "words": number(row.get("word_count")), "text": store(text(row.get("document_text"))), "model_text": store(text(row.get("policy_view_model"))), "profiles": {}}
        for layer in layers:
            cols = bt_columns if layer == "bt" else [f"lda_topic_{i}" if initial else f"lda_k{layer[-1]}_topic_{i}" for i in range(len(layers[layer]["topics"]))]
            source_frame = bt if layer == "bt" else ld
            values = as_topics(source_frame.loc[doc_id], cols) if doc_id in source_frame.index else {str(i): None for i in range(len(cols))}
            d["profiles"][layer] = values
        d["bt_outlier"] = number(bt.loc[doc_id].get("bt_outlier")) if initial and doc_id in bt.index else None
        if d["uid"] in {x["uid"] for x in records}:
            raise ValueError("Identidad documental ambigua: revisar fuente, provincia, año y título")
        records.append(d); doc_by_id[doc_id] = d

    chunk_records = []
    for _, row in chunks.iterrows():
        cid, did = str(row.chunk_id), str(row.doc_id)
        if initial:
            assignment = number(training.loc[cid].topic) if cid in training.index else None
            weights = None
            policy_text = text(policy.loc[cid].get("policy_view")) if cid in policy.index else ""
            basis = "entrenamiento exportado" if cid in training.index else "sin asignación temática exportada"
            citation = f"[{did}, {cid}]"
        else:
            assignment = number(row.get("substantive_topic"))
            weights = as_topics(row, bt_columns) if pd.notna(row.get(bt_columns[0])) else None
            policy_text = text(row.get("policy_view"))
            basis = "propagación sustantiva exportada" if assignment is not None else "fuera del nivel sustantivo"
            citation = text(row.get("citation")) or f"[{did}, {cid}]"
        chunk_records.append({"id": cid, "doc": did, "uid": doc_by_id[did]["uid"], "text": store(text(row.get("text_raw"))), "policy": store(policy_text), "heading": text(row.get("heading")), "page_start": number(row.get("page_start")), "page_end": number(row.get("page_end")), "article": text(row.get("article_ref")), "origin": text(row.get("text_origin")), "assignment": int(assignment) if assignment is not None else None, "assignment_basis": basis, "weights": weights, "citation": citation})

    arrays = []
    for name, expected in [("document_embeddings.npy" if initial else "rag_document_embeddings.npy", len(docs)), ("chunk_embeddings.npy" if initial else "rag_chunk_embeddings.npy", len(chunks))]:
        p = export.file(name, required=False)
        if p is not None:
            a = np.load(p, mmap_mode="r", allow_pickle=False)
            ok = a.ndim == 2 and len(a) == expected and bool(np.isfinite(a).all())
            if not ok:
                raise ValueError(f"Embeddings inválidos: {name}")
            arrays.append({"name": name, "shape": list(a.shape), "finite": True})
    return {"id": stage_id, "name": "Inicial" if initial else "Refinada", "export": source, "manifest": manifest, "documents": records, "chunks": chunk_records, "layers": layers, "exported_crosswalk": crosswalk, "robustness": robustness, "embedding_checks": arrays}


def conditional(values):
    vals = np.array([v if v is not None else np.nan for v in values], dtype=float)
    if not np.isfinite(vals).all() or vals.sum() <= 0:
        return None
    return vals / vals.sum()


def correspondence_table(payload, layer="bt"):
    a, b = payload["stages"]
    if layer not in a["layers"] or layer not in b["layers"]:
        return pd.DataFrame()
    ad = {d["uid"]: d for d in a["documents"] if d["full"] and d["compare"] and not d["regional"]}
    bd = {d["uid"]: d for d in b["documents"] if d["full"] and d["compare"] and not d["regional"]}
    common = sorted(ad.keys() & bd.keys())
    rows, aa, bb, accepted = [], [], [], []
    for uid in common:
        x = conditional(list(ad[uid]["profiles"][layer].values()))
        y = conditional(list(bd[uid]["profiles"][layer].values()))
        if x is not None and y is not None:
            aa.append(x); bb.append(y); accepted.append(uid)
    if not aa:
        return pd.DataFrame()
    aa, bb = np.asarray(aa), np.asarray(bb)
    for i in range(aa.shape[1]):
        for j in range(bb.shape[1]):
            denom = np.linalg.norm(aa[:, i]) * np.linalg.norm(bb[:, j])
            rows.append({"layer": layer, "initial_topic": i, "refined_topic": j, "document_cosine": float(np.dot(aa[:, i], bb[:, j]) / denom) if denom else None, "common_documents": len(accepted), "basis": "co-distribución documental; no probabilidad ni descendencia del tópico"})
    return pd.DataFrame(rows)


def build_payload(initial_path, refined_path, workdir):
    workdir = Path(workdir)
    workdir.mkdir(parents=True, exist_ok=True)
    texts = {}
    def store(value):
        if not value:
            return None
        key = hashlib.sha256(value.encode()).hexdigest()
        texts.setdefault(key, value)
        return key
    stages = []
    for stage_id, path in [("initial", initial_path), ("refined", refined_path)]:
        root, source = open_export(path, workdir / "inputs")
        stages.append(load_stage(root, stage_id, source, store))
    a, b = stages
    ad, bd = ({d["uid"]: d for d in s["documents"]} for s in stages)
    coverage = []
    for uid in sorted(ad.keys() | bd.keys()):
        x, y = ad.get(uid), bd.get(uid)
        d = y or x
        coverage.append({"uid": uid, "initial_doc": x["id"] if x else None, "refined_doc": y["id"] if y else None, "title": d["title"], "source": d["source"], "province": d["province"], "regional": d["regional"], "initial_fulltext": bool(x and x["full"]), "refined_fulltext": bool(y and y["full"]), "initial_origin": x["origin"] if x else None, "refined_origin": y["origin"] if y else None, "identity_basis": "fuente + provincia + año matriz + título normalizados"})
    ac = {(c["uid"], c["id"]): c for c in a["chunks"]}
    bc = {(c["uid"], c["id"]): c for c in b["chunks"]}
    lineage = []
    for key in sorted(ac.keys() | bc.keys()):
        x, y = ac.get(key), bc.get(key)
        unchanged = bool(x and y and x["text"] == y["text"])
        lineage.append({"uid": key[0], "chunk_id": key[1], "initial_present": x is not None, "refined_present": y is not None, "same_raw_text": unchanged if x and y else None, "status": "conservado" if unchanged else "modificado" if x and y else "incorporado" if y else "no presente en refinada", "initial_assignment": x["assignment"] if x else None, "refined_assignment": y["assignment"] if y else None, "refined_assignment_basis": y["assignment_basis"] if y else None})
    payload = {"schema": "observatorio-noa-panorama/0.1", "version": VERSION, "created_at": datetime.now(timezone.utc).isoformat(), "provinces": PROVINCES, "texts": texts, "stages": stages, "coverage": coverage, "lineage": lineage, "references": [{"title": "TopicPanorama: a Full Picture of Relevant Topics", "doi": "10.1109/VAST.2014.7042494", "url": "https://shixialiu.com/publications/TopicPanorama/index.htm"}], "limitations": ["Adaptación de funciones visuales: no replica los algoritmos CTM, BRT ni el emparejamiento húngaro del sistema original.", "Las relaciones entre etapas expresan co-distribución documental o coincidencia de fragmentos con asignación exportada; no son equivalencias certificadas.", "Inicial: los perfiles BERTopic se condicionan a la masa de los tres tópicos, excluyendo la masa atípica; refinada: se usan las afinidades documentales sustantivas normalizadas ya exportadas.", "Los descriptores refinados no incluyen pesos de términos LDA ni centroides originales. No se inventan esos valores.", "El eje de etapas representa versiones del procesamiento. Los años corresponden a los documentos.", "Los dos estados no permiten atribuir causalmente el cambio a cada intervención de procesamiento.", "Las similitudes, afinidades y métricas de estabilidad no son probabilidades calibradas de verdad."]}
    payload["correspondences"] = correspondence_table(payload).to_dict("records")
    payload["correspondences_lda3"] = correspondence_table(payload, "lda3").to_dict("records")
    return enrich_payload(payload)


def write_project(payload, template, output):
    payload = enrich_payload(payload)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    data_json = json.dumps(payload, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
    template = Path(template).read_text(encoding="utf-8")
    html = template.replace("__NOA_PAYLOAD__", data_json.replace("<", "\\u003c").replace("\u2028", "\\u2028").replace("\u2029", "\\u2029"))
    html_path = output / "Observatorio_NOA_TopicPanorama_Evolucion_v0_2.html"
    html_path.write_text(html, encoding="utf-8")
    (output / "panorama_datos.json").write_text(data_json, encoding="utf-8")
    pd.DataFrame(payload["coverage"]).to_csv(output / "auditoria_documentos.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(payload["lineage"]).to_csv(output / "auditoria_fragmentos.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(payload["correspondences"]).to_csv(output / "correspondencias_documentales_bt.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(payload["correspondences_lda3"]).to_csv(output / "correspondencias_documentales_lda3.csv", index=False, encoding="utf-8-sig")
    analysis_tables = export_analyses(payload, output)
    report = {"analytical_tables": analysis_tables, "reused_results": payload.get("reused_results"), "version": VERSION, "documents": [{"stage": s["id"], "catalogued": len(s["documents"]), "fulltext": sum(d["full"] for d in s["documents"]), "provincial_fulltext": sum(d["full"] and d["compare"] and not d["regional"] for d in s["documents"]), "chunks": len(s["chunks"]), "input": s["export"], "embedding_checks": s["embedding_checks"]} for s in payload["stages"]], "chunk_lineage": {status: sum(r["status"] == status for r in payload["lineage"]) for status in sorted({r["status"] for r in payload["lineage"]})}, "recovered_documents": [r for r in payload["coverage"] if not r["initial_fulltext"] and r["refined_fulltext"]], "limitations": payload["limitations"]}
    (output / "resumen_validacion.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return html_path, report

Writing noa_panorama.py


## 4. Escribir el visor autónomo
La interfaz está incluida en esta notebook y funciona sin bibliotecas JavaScript externas. Las tablas y los pasajes quedan incorporados al HTML.

In [5]:
%%writefile visor_template.html
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Observatorio NOA · Evolución de la evidencia</title>
<style>
:root{--ink:#273640;--muted:#65717a;--line:#dde2e5;--paper:#f5f6f6;--accent:#812b48;--gold:#b88b3c;--cat:#2878a8;--lr:#c38c3a;--sde:#62916b}
*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--ink);font:14px/1.5 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}button,input,select,textarea{font:inherit}button,select{cursor:pointer}button{border:1px solid var(--line);border-radius:7px;background:white;color:var(--ink);padding:7px 12px}button:hover{border-color:var(--accent);background:#fcf8fa}button.active{background:var(--accent);color:white;border-color:var(--accent)}button:disabled{opacity:.45;cursor:default}input,select,textarea{border:1px solid #cdd4d8;border-radius:6px;background:white;color:var(--ink);padding:7px;max-width:100%}input[type=checkbox]{accent-color:var(--accent)}input[type=range]{padding:0;accent-color:var(--accent)}textarea{width:100%;resize:vertical}a{color:var(--accent)}h1,h2,h3,p{margin-top:0}h1{font-size:29px;line-height:1.2;letter-spacing:-.6px;margin-bottom:8px}h2{font-size:18px;margin-bottom:8px}h3{font-size:14px;margin:16px 0 7px}.eyebrow{font-size:11px;letter-spacing:2px;font-weight:700;color:var(--accent);margin-bottom:8px}.muted{color:var(--muted)}.small{font-size:12px}.app{max-width:1800px;margin:auto;padding:26px 28px 36px}.header{display:flex;gap:20px;justify-content:space-between;align-items:flex-start;margin-bottom:20px}.header p{margin:0;color:var(--muted);max-width:820px}.badge{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--line);border-radius:20px;padding:3px 8px;font-size:11px;background:white;white-space:nowrap}.dot{width:9px;height:9px;border-radius:50%;display:inline-block}.legend{display:flex;gap:15px;flex-wrap:wrap;font-size:12px;margin-top:11px}.legend span{display:flex;gap:6px;align-items:center}.toolbar,.filters,.exportbar{display:flex;align-items:center;gap:8px;flex-wrap:wrap}.toolbar{justify-content:space-between;margin:12px 0}.segmented{display:inline-flex;gap:3px}.filters{background:white;padding:13px;border:1px solid var(--line);border-radius:10px;align-items:flex-end;margin-bottom:12px;gap:12px}.control{display:flex;flex-direction:column;gap:4px;font-size:11px;color:var(--muted)}.control label{font-weight:600}.control.search{flex:1;min-width:170px}.control select{font-size:12px;min-width:105px}.control input[type=search]{width:100%;font-size:12px}.check{display:flex;gap:5px;align-items:center;font-size:12px;color:var(--ink)}.settings{display:flex;gap:17px;align-items:center;flex-wrap:wrap;padding:0 2px 11px}.settings label{display:flex;gap:7px;align-items:center;font-size:12px}.settings select{font-size:12px;padding:4px}.workspace{display:grid;grid-template-columns:minmax(0,1fr) 325px;gap:15px;align-items:start}.panels{display:grid;gap:14px;grid-template-columns:1fr}.panels.compare{grid-template-columns:repeat(2,minmax(0,1fr))}.panel,.detail,.section-card{border:1px solid var(--line);background:white;border-radius:11px;overflow:hidden}.panel-head{padding:15px 16px 9px;border-bottom:1px solid #eef0f1}.panel-head h2{display:flex;justify-content:space-between;align-items:center;margin-bottom:4px}.panel-head .sub{font-size:12px;color:var(--muted)}.stats{display:grid;grid-template-columns:repeat(3,1fr);gap:5px;margin:11px 0 3px}.stat strong{font-size:22px;font-weight:600;display:block;line-height:1.2}.stat span{font-size:10px;color:var(--muted)}.diagram{display:block;width:100%;height:auto;background:#fff;touch-action:none;cursor:grab}.diagram:active{cursor:grabbing}.diagram text{font-family:Arial,sans-serif;fill:#33414a}.diagram .topic-label{font-size:12px;paint-order:stroke;stroke:white;stroke-width:4px;stroke-linejoin:round}.diagram .topic-id{font-size:13px;font-weight:700}.diagram .province-label{font-size:19px;font-weight:600}.diagram [data-topic],.diagram [data-doc],.diagram [data-chunk],.diagram [data-province]{cursor:pointer}.diagram .node:hover{filter:drop-shadow(0 0 3px #812b4870)}.diagram .selected{stroke:var(--accent)!important;stroke-width:4!important}.diagram .edge{stroke:#8c969e;stroke-opacity:.4}.panel-foot{padding:10px 15px;border-top:1px solid #eef0f1;font-size:11px;color:var(--muted);min-height:56px}.panel-foot p{margin:3px 0}.detail{position:sticky;top:12px;padding:17px;max-height:88vh;overflow:auto}.detail h2{font-size:17px}.detail .empty{color:var(--muted);padding:10px 0}.chips{display:flex;gap:5px;flex-wrap:wrap}.chip{font-size:11px;padding:3px 7px;border-radius:5px;background:#f0f3f5}.wordrow{display:grid;grid-template-columns:1fr 90px 42px;gap:7px;align-items:center;font-size:12px;margin:5px 0}.bar{height:5px;background:#e9edef;border-radius:3px}.bar i{display:block;height:100%;background:var(--accent);border-radius:3px}.docrow{border:1px solid var(--line);border-radius:7px;padding:9px;margin:7px 0;text-align:left;width:100%;background:white;cursor:pointer}.docrow strong{display:block;font-size:12px;font-weight:600;line-height:1.45}.docrow span{font-size:10px;color:var(--muted)}.docrow:hover{background:#f9f5f6}.notice{border-left:3px solid var(--gold);padding:8px 10px;background:#fcf8ef;font-size:12px;margin:10px 0}.notice.red{border-color:#a44747;background:#fcf2f2}.text-block{white-space:pre-wrap;font:12px/1.65 Georgia,serif;max-height:380px;overflow:auto;padding:11px;background:#f8f9f9;border:1px solid var(--line);border-radius:6px;overflow-wrap:anywhere}.chunk{border-top:1px solid var(--line);padding:10px 0}.chunk p{font:12px/1.6 Georgia,serif;margin:7px 0;white-space:pre-wrap}.chunk .meta{font-size:10px;color:var(--muted)}.detail-tabs{display:flex;gap:4px;flex-wrap:wrap;margin:10px 0}.detail-tabs button{font-size:11px;padding:4px 7px}.section-card{padding:20px;margin-top:14px}.section-card h2{margin-bottom:6px}.section-intro{font-size:13px;color:var(--muted);max-width:1000px}.section-card table{border-collapse:collapse;width:100%;font-size:12px}.section-card th,.section-card td{padding:8px;border-bottom:1px solid var(--line);text-align:left;vertical-align:top}.section-card th{font-weight:600;color:var(--muted)}.table-wrap{overflow:auto}.heatmap td,.heatmap th{text-align:center;min-width:58px}.heatmap td{cursor:pointer}.heatmap td:hover{outline:2px solid var(--accent);outline-offset:-2px}.heatmap .picked{outline:3px solid var(--accent);outline-offset:-3px}.corr-grid{display:grid;grid-template-columns:minmax(0,1fr) 300px;gap:15px}.corr-svg{width:100%;height:auto;max-height:570px}.audit-status{font-size:11px;font-weight:600}.recovered{color:#397047}.audit-buttons{display:flex;gap:7px;flex-wrap:wrap;margin:12px 0}.method-grid{display:grid;grid-template-columns:1fr 1fr;gap:28px}.method-grid p,.method-grid li{font-size:13px}.statusline{margin:10px 0 0;color:var(--muted);font-size:11px}.toast{position:fixed;bottom:22px;left:50%;transform:translateX(-50%);background:#243640;color:white;padding:10px 18px;border-radius:8px;z-index:10;box-shadow:0 3px 15px #0002;max-width:90vw}.hidden{display:none!important}.exportbar button{font-size:11px;padding:6px 9px}.footer{border-top:1px solid var(--line);margin-top:20px;padding-top:12px;font-size:11px;color:var(--muted)}.metricline{display:flex;gap:16px;flex-wrap:wrap;margin:12px 0}.metricline strong{font-size:22px;display:block}.metricline span{font-size:11px}.selected-note{font-size:11px;color:var(--accent);margin-bottom:9px}.group-tag{border-radius:4px;background:#f0e7eb;padding:2px 5px;font-size:10px;color:var(--accent)}
@media(max-width:1150px){.workspace{grid-template-columns:1fr}.detail{position:static;max-height:none}.detail .doclist{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:0 10px}.corr-grid{grid-template-columns:1fr}.method-grid{grid-template-columns:1fr}.app{padding:20px}.header{flex-direction:column}}@media(max-width:720px){.panels.compare{grid-template-columns:1fr}.app{padding:13px}.filters{gap:9px}.control{flex:1;min-width:110px}.control.search{min-width:100%}.detail .doclist{display:block}h1{font-size:25px}.settings{gap:10px}.exportbar{margin-top:8px}.section-card{padding:13px}}
@media print{body{background:white}.filters,.toolbar,.settings,.exportbar,.detail,.footer button{display:none!important}.app{padding:0}.workspace{display:block}.panels.compare{grid-template-columns:1fr 1fr}.panel{break-inside:avoid}.section-card{break-inside:avoid}.hidden{display:none!important}}
.scenario-strip{padding:9px 13px;background:#faf4e8;border:1px solid #e5d6b5;border-radius:8px;margin-bottom:12px;font-size:12px;display:flex;gap:8px;align-items:center;flex-wrap:wrap}.scenario-strip .excluded-chip{font-size:11px;padding:4px 7px;background:white;border-color:#d9c9ac}.analytic-tabs{display:flex;gap:5px;flex-wrap:wrap;margin:12px 0}.analytic-tabs button{font-size:12px}.analytic-controls{display:flex;gap:12px;align-items:flex-end;flex-wrap:wrap;margin:12px 0}.analytic-controls label{display:flex;flex-direction:column;gap:3px;font-size:11px;color:var(--muted)}.analytic-controls select{font-size:12px}.profiles-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:15px}.profile-card{border:1px solid var(--line);border-radius:8px;padding:14px}.profile-card h3{margin:0 0 5px;font-size:16px}.profile-card .profile-legend{display:flex;gap:12px;font-size:10px;margin:10px 0}.profile-row{display:grid;grid-template-columns:38px minmax(0,1fr) 42px 42px;gap:7px;align-items:center;margin:11px 0;font-size:11px}.profile-track{height:13px;position:relative;background:#f4f5f5;border-radius:2px}.profile-track i{position:absolute;left:0;height:5px;border-radius:2px}.profile-track .observed{top:0;background:#899aa5}.profile-track .balanced{bottom:0;background:#812b48}.profile-row button{padding:1px 2px;border:0;background:transparent;font-size:11px;text-align:left;color:var(--accent)}.profile-head{font-size:10px;color:var(--muted);margin-bottom:3px}.analytic-note{font-size:12px;color:var(--muted);margin:9px 0}.influence-summary{display:flex;gap:22px;flex-wrap:wrap;margin:15px 0}.influence-summary strong{display:block;font-size:25px;font-weight:600}.influence-summary span{font-size:11px;color:var(--muted)}.support-bar{width:110px;background:#f0f2f3;height:7px;border-radius:3px;display:inline-block;vertical-align:middle;margin-right:7px}.support-bar i{display:block;height:100%;background:var(--accent);border-radius:3px}.excluded-row{opacity:.6;background:#f5f5f5}.influence-table td{font-size:11px}.influence-table .document-title{max-width:360px}.influence-table .document-title button{text-align:left;border:0;background:transparent;padding:0;font-size:11px;color:var(--accent)}.lift-cell{cursor:pointer!important;min-width:65px!important;text-align:center!important}.lift-cell strong{display:block;font-size:13px}.lift-cell span{display:block;font-size:10px}.lift-mark{outline:2px solid #812b48;outline-offset:-3px}.validated-label{color:#397047;font-size:11px}.proposed-label{color:#936d2d;font-size:11px}.interpretation{margin:12px 0;padding:10px;border:1px solid #e9dfc8;background:#fdfaf3;border-radius:7px}.interpretation h3{margin-top:0}.analysis-actions{display:flex;gap:7px;flex-wrap:wrap;margin:10px 0}.analysis-actions button{font-size:11px}.review-ref{font-size:10px;padding:3px 5px;margin:3px}.reference-line{font-size:11px;margin:9px 0;color:var(--muted)}.help-inline{font-size:10px;line-height:1.4;color:var(--muted)}
@media(max-width:1000px){.profiles-grid{grid-template-columns:1fr}.profile-row{grid-template-columns:40px minmax(0,1fr) 55px 55px}.support-bar{width:65px}}

</style>
</head>
<body>
<main class="app">
 <header class="header"><div><div class="eyebrow">OBSERVATORIO NOA · CARTOGRAFÍA Y TRAZABILIDAD</div><h1>Cómo cambia la evidencia</h1><p>Explore las etapas inicial y refinada del análisis documental. Seleccione un tópico para seguir sus palabras, documentos y fragmentos.</p><div class="legend"><span><i class="dot" style="background:var(--cat)"></i>Catamarca</span><span><i class="dot" style="background:var(--lr)"></i>La Rioja</span><span><i class="dot" style="background:var(--sde)"></i>Santiago del Estero</span></div></div><span class="badge">Adaptación de TopicPanorama · v0.2</span></header>
 <div class="toolbar"><div class="segmented" id="view-buttons"><button data-view="initial">Inicial</button><button data-view="refined">Refinada</button><button data-view="compare" class="active">Comparar</button></div><div class="exportbar"><button id="save-svg">Exportar SVG</button><button id="save-png">Exportar PNG</button><button id="save-session">Guardar sesión</button><button id="load-session">Cargar sesión</button><input id="session-file" class="hidden" type="file" accept="application/json,.json"></div></div>
 <div class="filters">
  <div class="control"><label for="universe">Universo</label><select id="universe"><option value="all">Disponible en cada etapa</option><option value="common">Textos completos comunes</option></select></div>
  <div class="control"><label for="province">Provincia</label><select id="province"><option value="all">Las tres provincias</option><option>Catamarca</option><option>La Rioja</option><option>Santiago del Estero</option></select></div>
  <div class="control"><label for="source">Fuente</label><select id="source"><option value="all">CFI y SAIJ</option><option>CFI</option><option>SAIJ</option></select></div>
  <div class="control"><label for="period">Período documental</label><select id="period"><option value="all">Todos</option></select></div>
  <div class="control"><label for="layer">Capa temática</label><select id="layer"><option value="bt">BERTopic</option><option value="lda3">LDA K=3</option><option value="lda2">LDA K=2 · refinada</option></select></div>
  <div class="control search"><label for="query">Buscar en títulos y textos</label><input id="query" type="search" placeholder="Microcrédito, comunicación, software…"></div>
  <button id="reset">Restablecer</button>
 </div>
 <div class="settings"><label>Ponderación <select id="balance"><option value="documents">Igual peso por documento</option><option value="sources">Equilibrar CFI y SAIJ</option></select></label><label>Etiquetas <select id="label-mode"><option value="keywords">Palabras originales</option><option value="proposed">Propuestas interpretativas</option><option value="validated">Solo interpretaciones validadas</option></select></label><label>Detalle <select id="detail-level"><option value="topics">Tópicos</option><option value="documents">Documentos</option><option value="chunks">Fragmentos</option></select></label><label>Umbral de relación <input id="threshold" type="range" min="0" max="1" step=".01" value=".45"><span id="threshold-value">0,45</span></label><label><input type="checkbox" id="density" checked> Densidad visual</label><label><input type="checkbox" id="regional"> Contexto ATACALAR</label><button id="zoom-out" title="Alejar">−</button><button id="zoom-in" title="Acercar">+</button><button id="zoom-reset">Encuadrar</button></div>
 <div id="scenario-strip" class="scenario-strip"></div>
<div class="workspace"><div id="panels" class="panels compare"></div><aside id="detail" class="detail" aria-live="polite"></aside></div>
 <section class="section-card" id="provincial-analysis">
 <h2>Lectura provincial y respaldo documental</h2>
 <p class="section-intro">Compare ponderaciones, identifique qué documentos sostienen cada tópico y explore exclusiones reversibles con los perfiles ya calculados.</p>
 <div class="analytic-tabs"><button data-analysis-tab="profiles">Perfiles y fuentes</button><button data-analysis-tab="influence">Aporte y exclusión</button><button data-analysis-tab="specificity">Concentración relativa</button></div>
 <div class="analytic-controls"><label>Etapa<select id="analysis-stage"><option value="initial">Inicial</option><option value="refined">Refinada</option></select></label><label id="analysis-province-control">Provincia del respaldo<select id="analysis-province"><option>Catamarca</option><option selected>La Rioja</option><option>Santiago del Estero</option></select></label><label id="analysis-topic-control">Tópico<select id="analysis-topic"></select></label><label id="lift-threshold-control">Umbral exploratorio de concentración<input id="lift-threshold" type="number" min="1" max="3" step=".05" value="1.2" style="width:115px"></label><label id="support-min-control">Mínimo de documentos con tópico entre sus dos mayores afinidades<input id="support-min" type="number" min="1" max="19" step="1" value="2" style="width:90px"></label></div>
 <div class="analysis-actions"><button id="export-analysis-csv">Descargar tabla actual</button><button id="export-analysis-svg">Exportar figura SVG</button><button id="export-analysis-png">Exportar figura PNG</button></div>
 <div id="analysis-content"></div>
 <p class="analytic-note">Las exclusiones definen un escenario de lectura y se aplican a ambas etapas. Los promedios se recalculan con perfiles fijos; los ajustes de LDA y BERTopic se conservan. La auditoría mantiene todos los registros originales.</p>
</section>

<section class="section-card" id="correspondence-section"><h2>Correspondencias entre etapas</h2><p class="section-intro">Las relaciones pueden ser múltiples. Los números de tópico son propios de cada etapa. Seleccione una celda para consultar la evidencia y registrar una revisión.</p><div class="toolbar"><label class="small">Criterio <select id="correspondence-method"><option value="documents">Co-distribución documental</option><option value="chunks">Fragmentos con asignación exportada</option></select></label><button id="export-corr">Descargar tabla de correspondencias</button></div><div id="corr-summary" class="small muted"></div><div class="corr-grid"><div><svg id="corr-svg" class="corr-svg" viewBox="0 0 950 540" role="img" aria-label="Correspondencias entre etapas"></svg><div id="heatmap" class="table-wrap"></div></div><aside id="pair-detail"></aside></div></section>
 <section class="section-card" id="audit-section"><h2>Qué se conservó y qué se recuperó</h2><p class="section-intro">La identidad documental se verifica por fuente, provincia, año y título. La continuidad de un fragmento exige además el mismo identificador y texto original.</p><div id="audit-metrics" class="metricline"></div><div class="audit-buttons"><button id="export-docs">Descargar auditoría documental</button><button id="export-chunks">Descargar auditoría de fragmentos</button><button id="export-log">Descargar revisiones expertas</button></div><div id="audit-table" class="table-wrap"></div><details style="margin-top:14px"><summary>Registro del refinamiento del procesamiento</summary><div id="processing-table" class="table-wrap"></div></details></section>
 <section class="section-card" id="method-section"><h2>Cómo leer esta cartografía</h2><div class="method-grid"><div><h3>Representación</h3><p>Los sectores externos identifican provincias. Sus subdivisiones muestran el perfil temático de los documentos disponibles. Los círculos centrales representan tópicos; su tamaño crece con la afinidad media y sus porciones, la composición provincial con igual peso entre provincias disponibles. La posición combina esa composición con relaciones documentales y ajustes de legibilidad.</p><p>Las líneas expresan similitud coseno entre distribuciones de un tópico sobre documentos. El fondo representa densidad visual de nodos. Las distancias de la figura no constituyen una escala de impacto de políticas públicas.</p><h3>Dos alcances del análisis</h3><p>La etapa inicial presenta tres tópicos BERTopic globales. La refinada presenta ocho microtópicos sustantivos; su manifiesto conserva, además, un modelo global de dos tópicos. Sus diferencias de alcance permanecen identificadas.</p><p>En BERTopic se comparan perfiles documentales condicionados a masa temática: se normalizan los tres pesos iniciales sin la masa atípica; los ocho pesos refinados ya están normalizados en la exportación. Afinidad y similitud no son probabilidades calibradas.</p><h3>Balance de fuentes</h3><p>El modo equilibrado asigna el mismo peso a CFI y SAIJ dentro de cada provincia. Cuando falta una fuente, ese perfil queda no disponible; puede consultarse mediante la ponderación observada. ATACALAR se consulta en la auditoría y la evidencia como contexto regional; se excluye de los perfiles provinciales y de las correspondencias entre etapas.</p></div><div><h3>Correspondencias y revisión</h3><p>El criterio documental compara vectores sobre textos completos comunes y muestra similitud coseno. El criterio de fragmentos cuenta únicamente asignaciones presentes en ambas exportaciones, separando atípicos y fragmentos fuera del nivel sustantivo. En la etapa inicial esas asignaciones están exportadas solo para la muestra de entrenamiento.</p><p>Las revisiones y etiquetas expertas se guardan como anotaciones reversibles. Los pesos y asignaciones exportados permanecen disponibles. Las agrupaciones expertas identifican conjuntos interpretativos mediante su etiqueta compartida. Las etiquetas propuestas están identificadas; validarlas requiere fundamento y una referencia a evidencia exportada.</p><h3>Respaldo y concentración</h3><p>El aporte documental es la contribución ponderada de un documento dividida por la afinidad provincial del tópico. La exclusión recalcula promedios y relaciones con perfiles fijos. Si una exclusión elimina una fuente, el balance CFI/SAIJ queda no disponible.</p><p>La concentración relativa divide la afinidad provincial por el promedio de las provincias disponibles, todas con igual peso. El filtro de provincia conserva esa referencia. Se requieren dos provincias para interpretar el contraste. El soporte cuenta documentos donde el tópico figura entre las dos mayores afinidades, incluidos empates; en K=2 todos los documentos satisfacen ese criterio.</p><h3>Alcance de la versión</h3><ul id="limitations"></ul><h3>Referencia</h3><p>Liu, S., Wang, X., Chen, J., Zhu, J. y Guo, B. (2014). <em>TopicPanorama: a Full Picture of Relevant Topics.</em> IEEE VAST. <a href="https://doi.org/10.1109/VAST.2014.7042494" target="_blank" rel="noopener">DOI: 10.1109/VAST.2014.7042494</a>.</p><p class="small muted">El HTML funciona sin conexión. La notebook carga exportaciones existentes y genera este visor; la comparación entre etapas no estima por separado el efecto causal de cada intervención.</p></div></div><div id="provenance" class="statusline"></div></section>
 <footer class="footer">Observatorio de políticas productivas e innovación provincial · Catamarca, La Rioja y Santiago del Estero · Datos documentales 2009–2019. Las etapas representan versiones del procesamiento.</footer>
</main>
<div id="toast" class="toast hidden" role="status"></div>
<script id="noa-data" type="application/json">__NOA_PAYLOAD__</script>
<script>
"use strict";
const DATA=JSON.parse(document.getElementById('noa-data').textContent);
const $=id=>document.getElementById(id), NS='http://www.w3.org/2000/svg';
const COLORS={'Catamarca':'#2878a8','La Rioja':'#c38c3a','Santiago del Estero':'#62916b','Regional':'#929aa1'};
const ANGLES={'Catamarca':-Math.PI/2,'La Rioja':Math.PI/6,'Santiago del Estero':5*Math.PI/6,'Regional':Math.PI/2};
const fmt=(x,d=2)=>Number(x).toLocaleString('es-AR',{minimumFractionDigits:d,maximumFractionDigits:d});
const pct=x=>fmt(100*x,1)+' %';
const esc=x=>String(x??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const norm=x=>String(x??'').normalize('NFD').replace(/[\u0300-\u036f]/g,'').toLowerCase().replace(/_/g,' ');
const txt=key=>key?DATA.texts[key]||'':'';
const stages=Object.fromEntries(DATA.stages.map(s=>[s.id,s]));
const docMaps=Object.fromEntries(DATA.stages.map(s=>[s.id,Object.fromEntries(s.documents.map(d=>[d.id,d]))]));
const uidMaps=Object.fromEntries(DATA.stages.map(s=>[s.id,Object.fromEntries(s.documents.map(d=>[d.uid,d]))]));
const commonFull=new Set(DATA.coverage.filter(r=>r.initial_fulltext&&r.refined_fulltext).map(r=>r.uid));
const defaults={view:'compare',universe:'all',province:'all',source:'all',period:'all',layer:'bt',query:'',balance:'documents',detail:'topics',threshold:.45,density:true,regional:false,zoom:1,panX:0,panY:0,method:'documents'};
let state={...defaults},selection=null,pair=null,docTab='chunks',chunkPage=0,annotations={topics:{},pairs:{},log:[]},toastTimer;
const fingerprints=DATA.stages.map(s=>s.export.sha256||s.export.filename);
function toast(message){$('toast').textContent=message;$('toast').classList.remove('hidden');clearTimeout(toastTimer);toastTimer=setTimeout(()=>$('toast').classList.add('hidden'),4000);}
function svgNode(name,attrs={},children=[]){const n=document.createElementNS(NS,name);for(const[k,v]of Object.entries(attrs))if(v!==null&&v!==undefined)n.setAttribute(k,String(v));for(const c of children)n.append(typeof c==='string'?document.createTextNode(c):c);return n;}
function label(parent,x,y,value,attrs={}){const n=svgNode('text',{x,y,...attrs});n.textContent=value;parent.append(n);return n;}
function arc(cx,cy,r0,r1,a0,a1){const p=(r,a)=>[cx+r*Math.cos(a),cy+r*Math.sin(a)],a=p(r1,a0),b=p(r1,a1),c=p(r0,a1),d=p(r0,a0),large=a1-a0>Math.PI?1:0;return `M${a}A${r1},${r1},0,${large},1,${b}L${c}A${r0},${r0},0,${large},0,${d}Z`;}
function cosine(a,b,weights){let xy=0,xx=0,yy=0;for(let i=0;i<a.length;i++){const w=weights?weights[i]:1;xy+=w*a[i]*b[i];xx+=w*a[i]*a[i];yy+=w*b[i]*b[i];}return xx&&yy?Math.min(1,Math.max(0,xy/Math.sqrt(xx*yy))):null;}
function profile(doc,layer){const obj=doc.profiles[layer];if(!obj)return null;const vals=Object.values(obj);if(vals.some(x=>x===null||!Number.isFinite(x)))return null;const sum=vals.reduce((a,b)=>a+b,0);return sum>0?vals.map(x=>x/sum):null;}
function matches(doc,full=true){if(full&&!doc.full)return false;if(doc.regional&&!state.regional)return false;if(!doc.regional&&!doc.compare)return false;if(state.province!=='all'&&(doc.regional||doc.province!==state.province))return false;if(state.source!=='all'&&doc.source!==state.source)return false;if(state.period!=='all'&&doc.period!==state.period)return false;if(state.universe==='common'&&!commonFull.has(doc.uid))return false;if(state.query&&!norm(doc.title+' '+doc.summary+' '+txt(doc.text)).includes(norm(state.query)))return false;return true;}
function docsFor(stage,full=true){return stage.documents.filter(d=>matches(d,full));}
function groupProvince(doc){return doc.regional?'Regional':doc.province;}
function topicKey(sid,layer,tid){return `${sid}:${layer}:${tid}`;}
function topicName(sid,tid,layer=state.layer){const t=stages[sid].layers[layer]?.topics.find(t=>t.id===String(tid));const custom=annotations.topics[topicKey(sid,layer,tid)]?.label;return custom||t?.words.slice(0,2).map(w=>w.term.replace(/_/g,' ')).join(' · ')||('Tópico '+tid);}
function docWeights(docs){if(state.balance!=='sources'||state.source!=='all')return docs.map(()=>1);const counts={};for(const d of docs){const p=groupProvince(d);counts[p]??={};counts[p][d.source]=(counts[p][d.source]||0)+1;}return docs.map(d=>{const p=counts[groupProvince(d)],n=Object.keys(p).length;return 1/(Object.keys(counts).length*n*p[d.source]);});}
function aggregate(stage){const layer=stage.layers[state.layer];if(!layer)return {docs:[],topics:[],groups:[],edges:[],missing:[]};let docs=docsFor(stage).filter(d=>!d.regional&&profile(d,state.layer));const groups=[...DATA.provinces];const means={},missing=[];for(const p of groups){const ds=docs.filter(d=>groupProvince(d)===p);if(!ds.length){means[p]=null;continue;}if(state.balance==='sources'&&(!ds.some(d=>d.source==='CFI')||!ds.some(d=>d.source==='SAIJ'))){means[p]=null;for(const source of ['CFI','SAIJ'])if(!ds.some(d=>d.source===source))missing.push(`${p}: balance no disponible, falta texto completo ${source}`);continue;}const weights=docWeights(ds),total=weights.reduce((a,b)=>a+b,0);means[p]=layer.topics.map((t,i)=>ds.reduce((a,d,j)=>a+profile(d,state.layer)[i]*weights[j],0)/total);if(state.balance==='sources'&&state.source==='all'){for(const source of ['CFI','SAIJ'])if(!ds.some(d=>d.source===source))missing.push(`${p}: sin texto completo ${source} en esta vista`);}}
 docs=docs.filter(d=>means[groupProvince(d)]);const available=groups.filter(p=>means[p]);const topics=layer.topics.map((t,i)=>{const composition=groups.map(p=>means[p]?.[i]||0),sum=composition.reduce((a,b)=>a+b,0),shares=composition.map(v=>sum?v/sum:0),strength=available.length?sum/available.length:0;const ax=groups.reduce((v,p,j)=>v+Math.cos(ANGLES[p])*shares[j],0),ay=groups.reduce((v,p,j)=>v+Math.sin(ANGLES[p])*shares[j],0);const seed=(i*2.39996)+(stage.id==='initial'?.2:.7);return {id:t.id,words:t.words,shares,strength,x:380+160*ax+23*Math.cos(seed),y:340+160*ay+23*Math.sin(seed),anchorX:380+160*ax,anchorY:340+160*ay,r:17+48*Math.sqrt(strength)};});
 const weights=docWeights(docs),edges=[];for(let i=0;i<topics.length;i++)for(let j=i+1;j<topics.length;j++){const sim=cosine(docs.map(d=>profile(d,state.layer)[i]),docs.map(d=>profile(d,state.layer)[j]),weights);if(sim!==null&&sim>=state.threshold)edges.push({i,j,value:sim});}
 for(let k=0;k<100;k++){const forces=topics.map(t=>({x:(t.anchorX-t.x)*.045,y:(t.anchorY-t.y)*.045}));for(let i=0;i<topics.length;i++)for(let j=i+1;j<topics.length;j++){let dx=topics[i].x-topics[j].x,dy=topics[i].y-topics[j].y,dist=Math.hypot(dx,dy)||1;const target=topics[i].r+topics[j].r+45;let f=Math.max(0,target-dist)*.24+400/(dist*dist);forces[i].x+=dx/dist*f;forces[i].y+=dy/dist*f;forces[j].x-=dx/dist*f;forces[j].y-=dy/dist*f;}for(const e of edges){const a=topics[e.i],b=topics[e.j],dx=b.x-a.x,dy=b.y-a.y,dist=Math.hypot(dx,dy)||1,f=(dist-145)*.008*e.value;forces[e.i].x+=dx/dist*f;forces[e.i].y+=dy/dist*f;forces[e.j].x-=dx/dist*f;forces[e.j].y-=dy/dist*f;}for(let i=0;i<topics.length;i++){const t=topics[i];t.x+=forces[i].x;t.y+=forces[i].y;let dx=t.x-380,dy=t.y-340,dist=Math.hypot(dx,dy),limit=216-t.r;if(dist>limit){t.x=380+dx/dist*limit;t.y=340+dy/dist*limit;}}}
 return {docs,topics,groups,means,edges,missing};}
function placeTopicLabel(parent,t,name,boxes,nodes){const parts=name.split(' · ').slice(0,2).map(v=>v.length>24?v.slice(0,23)+'…':v),width=Math.max(...parts.map(v=>v.length))*6.25+6,height=parts.length*14;let best=null;for(const angle of [Math.PI/2,-Math.PI/2,0,Math.PI,Math.PI/4,3*Math.PI/4,-Math.PI/4,-3*Math.PI/4]){const ux=Math.cos(angle),uy=Math.sin(angle),distance=t.r+Math.abs(ux)*width/2+Math.abs(uy)*height/2+10,cx=t.x+ux*distance,cy=t.y+uy*distance,box={x:cx-width/2,y:cy-height/2,w:width,h:height};let score=distance*.05;for(const n of nodes){if(n.id===t.id)continue;const x=Math.max(box.x,Math.min(n.x,box.x+width)),y=Math.max(box.y,Math.min(n.y,box.y+height)),over=n.r+6-Math.hypot(n.x-x,n.y-y);if(over>0)score+=2000+over*200;}for(const b of boxes){const dx=Math.max(0,Math.min(box.x+width,b.x+b.w)-Math.max(box.x,b.x)),dy=Math.max(0,Math.min(box.y+height,b.y+b.h)-Math.max(box.y,b.y));score+=dx*dy*5;}for(const[x,y]of [[box.x,box.y],[box.x+width,box.y],[box.x,box.y+height],[box.x+width,box.y+height]])score+=Math.max(0,Math.hypot(x-380,y-340)-242)*10000;if(best===null||score<best.score)best={...box,cx,cy,score};}boxes.push(best);if(Math.hypot(best.cx-t.x,best.cy-t.y)>t.r+35)parent.append(svgNode('line',{x1:t.x,y1:t.y,x2:best.cx,y2:best.cy,stroke:'#9aa5ac','stroke-opacity':.45,'stroke-width':.7}));parts.forEach((v,i)=>label(parent,best.cx,best.y+12+i*14,v,{class:'topic-label','text-anchor':'middle'}));}
function densityImage(topics){const c=document.createElement('canvas');c.width=128;c.height=128;const ctx=c.getContext('2d'),im=ctx.createImageData(128,128),vals=new Float32Array(128*128);let max=0;for(let y=0;y<128;y++)for(let x=0;x<128;x++){const px=x/127*760,py=y/127*700;let v=0;for(const t of topics)v+=t.strength*Math.exp(-((px-t.x)**2+(py-t.y)**2)/(2*76**2));vals[y*128+x]=v;max=Math.max(max,v);}for(let i=0;i<vals.length;i++){const q=max?Math.floor(vals[i]/max*6):0;im.data[i*4]=120;im.data[i*4+1]=129;im.data[i*4+2]=137;im.data[i*4+3]=q*6;}ctx.putImageData(im,0,0);return c.toDataURL();}
function selectTopic(sid,tid){selection={stage:sid,kind:'topic',id:String(tid),layer:state.layer};docTab='chunks';chunkPage=0;state.detail='documents';syncControls();renderAll();}
function selectDoc(sid,did){selection={stage:sid,kind:'doc',id:did,uid:docMaps[sid][did].uid,layer:state.layer};docTab='chunks';chunkPage=0;renderAll();}
function selectChunk(sid,cid){const c=stages[sid].chunks.find(c=>c.id===cid);if(!c)return;selection={stage:sid,kind:'chunk',id:cid,doc:c.doc,uid:c.uid,layer:state.layer};renderAll();}
function focusTopic(stage){if(selection?.kind==='topic'&&selection.layer===state.layer){if(selection.stage===stage.id)return selection.id;if(state.layer==='bt'||state.layer==='lda3'){const corr=documentCorrespondences().items.filter(e=>selection.stage==='initial'?e.i===selection.id:e.j===selection.id).sort((a,b)=>b.value-a.value);if(corr.length)return stage.id==='initial'?corr[0].i:corr[0].j;}}return null;}
function renderPanorama(stage){const ag=aggregate(stage),catalog=docsFor(stage,false),available=docsFor(stage),ids=new Set(available.map(d=>d.id)),chunks=stage.chunks.filter(c=>ids.has(c.doc));const panel=document.createElement('article');panel.className='panel';panel.dataset.stage=stage.id;panel.innerHTML=`<div class="panel-head"><h2>${esc(stage.name)} <span class="badge">${stage.id==='initial'?'outputs_1821':'RAG v3.0.2'}</span></h2><div class="sub">${esc(stage.layers[state.layer]?.name||'Capa no exportada en esta etapa')}</div><div class="stats"><div class="stat"><strong>${available.length}<span class="small"> / ${catalog.length}</span></strong><span>textos completos / catalogados</span></div><div class="stat"><strong>${chunks.length.toLocaleString('es-AR')}</strong><span>fragmentos en esta vista</span></div><div class="stat"><strong>${ag.topics.length}</strong><span>tópicos de la capa</span></div></div></div>`;
 const svg=svgNode('svg',{viewBox:'0 0 760 700',class:'diagram',id:'panorama-'+stage.id,role:'img','aria-label':`Panorama ${stage.name}`});const defs=svgNode('defs'),clip=svgNode('clipPath',{id:'clip-'+stage.id});clip.append(svgNode('circle',{cx:380,cy:340,r:247}));defs.append(clip);svg.append(defs);
 const scene=svgNode('g',{'clip-path':`url(#clip-${stage.id})`});const moving=svgNode('g',{transform:`translate(${380+state.panX} ${340+state.panY}) scale(${state.zoom}) translate(-380 -340)`});scene.append(moving);svg.append(scene);if(state.density&&ag.topics.length)moving.append(svgNode('image',{href:densityImage(ag.topics),x:0,y:0,width:760,height:700}));
 for(const e of ag.edges){const a=ag.topics[e.i],b=ag.topics[e.j];const edge=svgNode('line',{x1:a.x,y1:a.y,x2:b.x,y2:b.y,class:'edge','stroke-width':.8+2.3*e.value});edge.append(svgNode('title',{},[`Similitud documental: ${fmt(e.value,3)}`]));moving.append(edge);}
 const focused=focusTopic(stage);const selectedDoc=selection?.uid?uidMaps[stage.id][selection.uid]:null;
 for(const p of ag.groups){const a=ANGLES[p],start=a-Math.PI*.3,end=a+Math.PI*.3,color=COLORS[p],has=!!ag.means?.[p];const ring=svgNode('path',{d:arc(380,340,282,294,start,end),fill:color,'fill-opacity':has?1:.18,'data-province':p});ring.append(svgNode('title',{},[`${p}: ${ag.docs.filter(d=>groupProvince(d)===p).length} documentos con perfil disponible`]));ring.addEventListener('click',()=>{state.province=p==='Regional'?'all':(state.province===p?'all':p);if(p==='Regional')state.regional=true;syncControls();renderAll();});svg.append(ring);let cursor=start;for(let i=0;i<ag.topics.length;i++){const fraction=ag.means?.[p]?.[i]||0;if(!fraction)continue;const finish=cursor+(end-start)*fraction;const ta=svgNode('path',{d:arc(380,340,263,279,cursor+.003,Math.max(cursor+.003,finish-.003)),fill:color,'fill-opacity':focused===ag.topics[i].id?1:.45+.35*fraction,stroke:'#fff','stroke-width':.7,'data-topic':ag.topics[i].id});ta.append(svgNode('title',{},[`${p} · T${ag.topics[i].id}: ${pct(fraction)} de afinidad media`]));ta.addEventListener('click',()=>selectTopic(stage.id,ag.topics[i].id));svg.append(ta);cursor=finish;}
 const lx=p==='Catamarca'?380:p==='La Rioja'?655:115,ly=p==='Catamarca'?20:650;label(svg,lx,ly,p,{class:'province-label','text-anchor':'middle',fill:color});}
 if((focused!==null||selectedDoc)&&ag.docs.length){const pd=selectedDoc?ag.docs.filter(d=>d.uid===selectedDoc.uid):ag.docs;for(const p of ag.groups){const ds=pd.filter(d=>groupProvince(d)===p);if(!ds.length)continue;const a=ANGLES[p],start=a-Math.PI*.3,end=a+Math.PI*.3;const weights=ds.map(d=>focused!==null?profile(d,state.layer)?.[Number(focused)]||0:1),sum=weights.reduce((a,b)=>a+b,0)||1;let cursor=start;for(let i=0;i<ds.length;i++){const finish=cursor+(end-start)*weights[i]/sum;const path=svgNode('path',{d:arc(380,340,248,260,cursor+.003,Math.max(cursor+.003,finish-.003)),fill:COLORS[p],'fill-opacity':selectedDoc?.uid===ds[i].uid?1:.55,'data-doc':ds[i].id});path.append(svgNode('title',{},[ds[i].title]));path.addEventListener('click',()=>selectDoc(stage.id,ds[i].id));svg.append(path);cursor=finish;}}}
 const groups={};for(const t of ag.topics){const custom=annotations.topics[topicKey(stage.id,state.layer,t.id)];if(custom?.group)(groups[custom.group]??=[]).push(t);}for(const[name,ts]of Object.entries(groups)){if(ts.length<2)continue;const xs=ts.map(t=>t.x),ys=ts.map(t=>t.y),x=Math.min(...xs)-48,y=Math.min(...ys)-48,w=Math.max(...xs)-x+48,h=Math.max(...ys)-y+48;moving.append(svgNode('rect',{x,y,width:w,height:h,rx:30,fill:'#812b48','fill-opacity':.025,stroke:'#812b48','stroke-opacity':.3,'stroke-dasharray':'5 5'}));label(moving,x+10,y+16,name,{'font-size':11,fill:'#812b48'});}
 if(state.detail!=='topics'&&ag.docs.length){let ds=[...ag.docs];if(selectedDoc)ds=ds.filter(d=>d.uid===selectedDoc.uid);else if(focused!==null)ds.sort((a,b)=>(profile(b,state.layer)?.[Number(focused)]||0)-(profile(a,state.layer)?.[Number(focused)]||0));ds=ds.slice(0,selectedDoc?1:8);for(let i=0;i<ds.length;i++){const d=ds[i],pp=profile(d,state.layer),best=focused!==null?Number(focused):pp.indexOf(Math.max(...pp)),parent=ag.topics[best];if(!parent)continue;const provincialDocs=ds.filter(x=>groupProvince(x)===groupProvince(d)),localIndex=provincialDocs.findIndex(x=>x.id===d.id),theta=ANGLES[groupProvince(d)]+(localIndex-(provincialDocs.length-1)/2)*(Math.PI*.46/Math.max(1,provincialDocs.length)),docRadius=state.detail==='chunks'?199:215,x=380+docRadius*Math.cos(theta),y=340+docRadius*Math.sin(theta);moving.append(svgNode('line',{x1:parent.x,y1:parent.y,x2:x,y2:y,stroke:COLORS[groupProvince(d)],'stroke-opacity':.4,'stroke-width':1}));const node=svgNode('g',{class:'node','data-doc':d.id});node.append(svgNode('circle',{cx:x,cy:y,r:10,fill:'#fff',stroke:COLORS[groupProvince(d)],'stroke-width':selectedDoc?.uid===d.uid?4:2}));label(node,x,y+22,d.id.replace('DOC_','D'),{'font-size':11,'text-anchor':'middle'});node.append(svgNode('title',{},[d.title]));node.addEventListener('click',()=>selectDoc(stage.id,d.id));moving.append(node);
 if(state.detail==='chunks'){let cs=chunks.filter(c=>c.doc===d.id);if(focused!==null&&state.layer==='bt')cs=cs.filter(c=>c.assignment===Number(focused));cs=cs.slice(0,selectedDoc?16:3);for(let j=0;j<cs.length;j++){const phi=theta+(j-(cs.length-1)/2)*.21,cx=x+35*Math.cos(phi),cy=y+35*Math.sin(phi),ch=cs[j];moving.append(svgNode('line',{x1:x,y1:y,x2:cx,y2:cy,stroke:'#b0b7bb','stroke-width':.6}));const dot=svgNode('circle',{cx,cy,r:4.5,fill:COLORS[groupProvince(d)],'data-chunk':ch.id});dot.append(svgNode('title',{},[`${ch.id} · ${ch.assignment_basis}`]));dot.addEventListener('click',()=>selectChunk(stage.id,ch.id));moving.append(dot);}}}}
 const labelBoxes=[...moving.querySelectorAll('g[data-doc] circle')].map(n=>({x:Number(n.getAttribute('cx'))-25,y:Number(n.getAttribute('cy'))-12,w:50,h:39}));for(const t of ag.topics){if(!t.strength)continue;const g=svgNode('g',{class:'node','data-topic':t.id});let cursor=-Math.PI/2;for(let i=0;i<t.shares.length;i++){const v=t.shares[i];if(v<=0)continue;const end=cursor+Math.PI*2*v;if(v>.99999)g.append(svgNode('circle',{cx:t.x,cy:t.y,r:t.r,fill:COLORS[ag.groups[i]]}));else g.append(svgNode('path',{d:arc(t.x,t.y,0,t.r,cursor,end),fill:COLORS[ag.groups[i]]}));cursor=end;}g.append(svgNode('circle',{cx:t.x,cy:t.y,r:t.r,fill:'none',stroke:focused===t.id?'#812b48':'#ffffff','stroke-width':focused===t.id?4:2}));label(g,t.x,t.y+4,'T'+t.id,{class:'topic-id','text-anchor':'middle',fill:'#fff',style:'fill:white;paint-order:stroke;stroke:#33414a;stroke-width:2px'});const name=topicName(stage.id,t.id);placeTopicLabel(g,t,name,labelBoxes,ag.topics);g.append(svgNode('title',{},[`T${t.id} · ${name}\nAfinidad media: ${pct(t.strength)}\n${ag.groups.map((p,i)=>p+': '+pct(t.shares[i])).join('\n')}`]));g.addEventListener('click',()=>selectTopic(stage.id,t.id));moving.append(g);}
 if(!ag.docs.length||!ag.topics.length){label(svg,380,335,stage.layers[state.layer]?(state.balance==='sources'?'Balance no disponible para estos filtros':'Sin evidencia para estos filtros'):'Capa no exportada en esta etapa',{'text-anchor':'middle','font-size':20,fill:'#65717a'});if(!ag.docs.length&&state.source==='SAIJ'&&state.province==='Santiago del Estero'&&stage.id==='initial')label(svg,380,365,'Los textos completos se recuperaron en la etapa refinada',{'text-anchor':'middle','font-size':13});}
 label(svg,380,679,state.detail==='topics'?'Seleccione un tópico o un sector provincial':'Detalle: hasta 8 documentos; amplíe la evidencia en el panel',{'text-anchor':'middle','font-size':12,fill:'#65717a'});panel.append(svg);
 const foot=document.createElement('div');foot.className='panel-foot';foot.innerHTML=`<p>${esc(state.balance==='sources'?'Equilibrio CFI/SAIJ; perfiles sin ambas fuentes se indican como no disponibles.':'Igual peso por documento dentro de cada provincia.')} Sectores provinciales de igual amplitud.</p>${ag.missing.map(m=>`<p class="notice">${esc(m)}</p>`).join('')}<p>${stage.id==='initial'?'BERTopic inicial: asignaciones de fragmento disponibles para la muestra de entrenamiento.':'BERTopic sustantivo: afinidades y asignaciones consultadas según disponibilidad exportada.'}</p>`;panel.append(foot);
 let dragging=false,startX=0,startY=0,baseX=0,baseY=0,moved=false;svg.addEventListener('pointerdown',ev=>{if(ev.target.closest('[data-topic],[data-doc],[data-chunk],[data-province]'))return;dragging=true;moved=false;startX=ev.clientX;startY=ev.clientY;baseX=state.panX;baseY=state.panY;svg.setPointerCapture(ev.pointerId);});svg.addEventListener('pointermove',ev=>{if(!dragging)return;const scale=760/svg.getBoundingClientRect().width;state.panX=baseX+(ev.clientX-startX)*scale;state.panY=baseY+(ev.clientY-startY)*scale;moved=true;moving.setAttribute('transform',`translate(${380+state.panX} ${340+state.panY}) scale(${state.zoom}) translate(-380 -340)`);});svg.addEventListener('pointerup',()=>{if(dragging&&moved)renderAll();dragging=false;});svg.addEventListener('wheel',ev=>{ev.preventDefault();state.zoom=Math.min(3,Math.max(.65,state.zoom*(ev.deltaY<0?1.12:1/1.12)));renderAll();},{passive:false});return panel;}
function getSelectionStage(){return stages[selection?.stage]||stages[state.view==='initial'?'initial':'refined'];}
function textWithMore(value){if(value.length<12000)return `<div class="text-block">${esc(value||'Sin texto exportado para esta vista.')}</div>`;return `<div class="text-block">${esc(value.slice(0,12000))}</div><details><summary class="small">Abrir texto completo (${value.length.toLocaleString('es-AR')} caracteres)</summary><div class="text-block">${esc(value)}</div></details>`;}
function safeLink(value){try{const u=new URL(value);return ['http:','https:'].includes(u.protocol)?u.href:null;}catch{return null;}}
function docButtons(stage,docs,topicIndex){return docs.map(d=>{const p=profile(d,state.layer),weight=topicIndex===undefined?null:p?.[topicIndex];return `<button class="docrow" data-open-doc="${esc(d.id)}"><strong>${esc(d.title)}</strong><span>${esc(groupProvince(d))} · ${esc(d.source)} · ${d.year||''}${weight===null?'':' · afinidad '+pct(weight)}</span></button>`;}).join('');}
function wireDocs(container,stage){container.querySelectorAll('[data-open-doc]').forEach(b=>b.onclick=()=>selectDoc(stage.id,b.dataset.openDoc));}
function renderDetail(){const box=$('detail');if(!selection){box.innerHTML=`<h2>De la figura a la evidencia</h2><p class="empty">Seleccione un tópico para consultar sus descriptores, asociaciones LDA y documentos. Seleccione un documento para seguir sus fragmentos en ambas etapas.</p><div class="notice">La etapa inicial y la refinada tienen alcances temáticos diferentes. Las correspondencias se exploran en el panel inferior.</div><h3>Cobertura global</h3><p class="small">${DATA.stages.map(s=>`${s.name}: ${s.documents.filter(d=>d.full).length}/${s.documents.length} textos completos; ${s.chunks.length.toLocaleString('es-AR')} fragmentos`).join('<br>')}</p><h3>Controles</h3><p class="small muted">La rueda y los botones +/− acercan la red. Arrastre el centro para desplazarla. Los filtros y el encuadre se sincronizan entre etapas.</p>`;return;}
 const stage=getSelectionStage();if(selection.kind==='topic'){const layer=stage.layers[selection.layer];const topic=layer?.topics.find(t=>t.id===selection.id);if(!topic){selection=null;renderDetail();return;}const tid=Number(topic.id),custom=annotations.topics[topicKey(stage.id,selection.layer,topic.id)]||{},ds=docsFor(stage).filter(d=>profile(d,selection.layer)).sort((a,b)=>profile(b,selection.layer)[tid]-profile(a,selection.layer)[tid]);box.innerHTML=`<div class="selected-note">${esc(stage.name)} · ${esc(layer.name)}</div><h2>T${esc(topic.id)} · ${esc(topicName(stage.id,topic.id,selection.layer))}</h2>${custom.group?`<span class="group-tag">${esc(custom.group)}</span>`:''}<h3>Palabras del modelo</h3>${topic.words.map(w=>w.weight!==null?`<div class="wordrow"><span>${esc(w.term.replace(/_/g,' '))}</span><div class="bar"><i style="width:${Math.min(100,w.weight/Math.max(...topic.words.map(w=>w.weight||0))*100)}%"></i></div><span>${fmt(w.weight,3)}</span></div>`:`<span class="chip" style="display:inline-block;margin:2px">${esc(w.term.replace(/_/g,' '))}</span>`).join('')}<p class="small muted">${layer.weighted_terms?'Pesos LDA originales para los términos exportados.':'Descriptores exportados; este paquete no incluye pesos individuales de las palabras.'}</p><div id="lda-associations"></div><h3>Documentos en esta vista (${ds.length})</h3><div class="doclist">${docButtons(stage,ds.slice(0,8),tid)}</div>${ds.length>8?'<p class="small muted">Se muestran los ocho documentos de mayor afinidad.</p>':''}<div id="stability-detail"></div><details><summary>Etiqueta y nota experta</summary><p class="small muted">La anotación queda asociada a esta etapa y tópico.</p><label class="small">Etiqueta<input id="topic-label" maxlength="90" style="width:100%" value="${esc(custom.label||'')}"></label><label class="small">Grupo interpretativo<input id="topic-group" maxlength="80" style="width:100%" value="${esc(custom.group||'')}"></label><label class="small">Nota<textarea id="topic-note" rows="3" maxlength="2000">${esc(custom.note||'')}</textarea></label><button id="save-topic">Guardar anotación</button> <button id="clear-topic">Restaurar descriptor</button></details>`;
 wireDocs(box,stage);if(selection.layer==='bt'){const ldLayer=stage.id==='initial'?'lda3':'lda2',ld=stage.layers[ldLayer];const valid=ds.filter(d=>profile(d,ldLayer));const den=valid.reduce((a,d)=>a+profile(d,'bt')[tid],0);$('lda-associations').innerHTML=`<h3>Asociaciones con ${ld.name}</h3>${ld.topics.map((t,j)=>{const v=den?valid.reduce((a,d)=>a+profile(d,'bt')[tid]*profile(d,ldLayer)[j],0)/den:null;return `<div class="small">LDA ${t.id}: ${v===null?'sin evidencia':fmt(v,3)} · ${esc(t.words.slice(0,3).map(w=>w.term.replace(/_/g,' ')).join(', '))}</div>`;}).join('')}<p class="small muted">Correspondencia recalculada para estos filtros; ponderación por afinidad documental, sin asignar una jerarquía entre modelos.</p>`;}
 const robust=stage.robustness.filter(r=>r.left_run==='bertopic_policy_cap8'&&Number(r.left_topic)===tid);$('stability-detail').innerHTML=selection.layer==='bt'?`<h3>Estabilidad disponible</h3><p class="small">${robust.length?'Robustez entre variantes exportada: '+fmt(robust[0].semantic_topic_robustness,3):'Este paquete no exporta una métrica de estabilidad específica para este tópico.'}</p>`:'';
 $('save-topic').onclick=()=>{const key=topicKey(stage.id,selection.layer,topic.id),before=annotations.topics[key]||null,after={label:$('topic-label').value.trim(),group:$('topic-group').value.trim(),note:$('topic-note').value.trim()};annotations.topics[key]=after;logEdit('topic',key,before,after);renderAll();toast('Anotación registrada. Guardá la sesión para conservarla.');};$('clear-topic').onclick=()=>{const key=topicKey(stage.id,selection.layer,topic.id);logEdit('topic',key,annotations.topics[key]||null,null);delete annotations.topics[key];renderAll();};return;}
 const doc=docMaps[stage.id][selection.kind==='chunk'?selection.doc:selection.id];if(!doc){selection=null;renderDetail();return;}const counterpart=uidMaps[stage.id==='initial'?'refined':'initial'][doc.uid];const cs=stage.chunks.filter(c=>c.doc===doc.id),start=chunkPage*10,visible=selection.kind==='chunk'?cs.filter(c=>c.id===selection.id):cs.slice(start,start+10);const origin=doc.full?(doc.origin==='ocr_text'?'Texto recuperado por OCR':'Texto completo'):'Resumen curado; texto completo pendiente';box.innerHTML=`<div class="selected-note">${esc(stage.name)} · ${esc(doc.id)}</div><h2>${esc(doc.title)}</h2><div class="chips"><span class="chip">${esc(groupProvince(doc))}</span><span class="chip">${esc(doc.source)}</span><span class="chip">${doc.year||''}</span></div><p class="small muted" style="margin-top:8px">${esc(origin)} · ${cs.length} fragmentos</p>${safeLink(doc.link)?`<a class="small" href="${esc(safeLink(doc.link))}" target="_blank" rel="noopener">Abrir fuente documental</a>`:''}${counterpart?`<button id="other-stage" style="display:block;margin-top:9px;font-size:11px">Consultar este documento en ${stage.id==='initial'?'Refinada':'Inicial'}</button>`:''}<div class="detail-tabs"><button data-doctab="chunks" class="${docTab==='chunks'?'active':''}">Fragmentos</button><button data-doctab="original" class="${docTab==='original'?'active':''}">Texto original</button><button data-doctab="model" class="${docTab==='model'?'active':''}">Vista del modelado</button></div><div id="doc-content"></div>`;
 if(counterpart)$('other-stage').onclick=()=>{const sid=stage.id==='initial'?'refined':'initial';selectDoc(sid,counterpart.id);};box.querySelectorAll('[data-doctab]').forEach(b=>b.onclick=()=>{docTab=b.dataset.doctab;renderDetail();});if(docTab==='original'){$('doc-content').innerHTML=(!doc.full?'<div class="notice">Esta etapa utiliza el resumen curado. La ausencia de texto completo no representa ausencia de política.</div>':'')+textWithMore(doc.full?txt(doc.text):doc.summary);return;}if(docTab==='model'){$('doc-content').innerHTML='<p class="small muted">Vista documental de modelado guardada en esta exportación.</p>'+textWithMore(txt(doc.model_text));return;}
 $('doc-content').innerHTML=visible.length?visible.map(c=>`<article class="chunk"><div class="meta">${esc(c.id)} ${c.page_start!==null?'· p. '+c.page_start+(c.page_end!==c.page_start?'–'+c.page_end:''):''}${c.article?' · '+esc(c.article):''}</div><div class="meta">${esc(c.assignment_basis)}${c.assignment!==null?' · '+(c.assignment<0?'atípico':'T'+c.assignment):''}</div><p>${esc(txt(c.text))}</p><div class="meta">${esc(c.citation)}</div><button class="small" data-copy-citation="${esc(c.citation)}">Copiar referencia</button>${txt(c.policy)?`<details><summary class="small">Vista estructurada exportada</summary><div class="text-block">${esc(txt(c.policy))}</div></details>`:''}</article>`).join(''):'<div class="notice">No hay fragmentos de texto completo en esta etapa para este documento.</div>';
 if(selection.kind!=='chunk'&&cs.length>10){$('doc-content').insertAdjacentHTML('beforeend',`<div class="toolbar"><button id="chunks-prev" ${chunkPage===0?'disabled':''}>Anterior</button><span class="small">${start+1}–${Math.min(start+10,cs.length)} de ${cs.length}</span><button id="chunks-next" ${start+10>=cs.length?'disabled':''}>Siguiente</button></div>`);$('chunks-prev').onclick=()=>{chunkPage--;renderDetail();};$('chunks-next').onclick=()=>{chunkPage++;renderDetail();};}box.querySelectorAll('[data-copy-citation]').forEach(b=>b.onclick=async()=>{try{await navigator.clipboard.writeText(b.dataset.copyCitation);toast('Referencia copiada');}catch{toast(b.dataset.copyCitation);}});}
function documentCorrespondences(){const a=stages.initial,b=stages.refined,la=a.layers[state.layer],lb=b.layers[state.layer];if(!la||!lb)return {items:[],left:[],right:[],n:0,note:'Esta capa no está exportada en ambas etapas.'};const ads=docsFor(a).filter(d=>!d.regional&&profile(d,state.layer)),bds=new Map(docsFor(b).filter(d=>!d.regional&&profile(d,state.layer)).map(d=>[d.uid,d]));const ds=ads.filter(d=>bds.has(d.uid)),weights=docWeights(ds),items=[];for(let i=0;i<la.topics.length;i++)for(let j=0;j<lb.topics.length;j++){const av=ds.map(d=>profile(d,state.layer)[i]),bv=ds.map(d=>profile(bds.get(d.uid),state.layer)[j]),value=cosine(av,bv,weights);if(value!==null)items.push({i:String(i),j:String(j),value,n:ds.length,evidence:ds.map((d,k)=>({uid:d.uid,doc:d.id,value:av[k]*bv[k]})).sort((x,y)=>y.value-x.value)});}return {items,left:la.topics.map(t=>({id:t.id,name:topicName('initial',t.id)})),right:lb.topics.map(t=>({id:t.id,name:topicName('refined',t.id)})),n:ds.length,note:`${ds.length} textos completos comunes · similitud coseno de distribuciones documentales normalizadas.`};}
function chunkCorrespondences(){if(state.layer!=='bt')return {items:[],left:[],right:[],n:0,note:'El criterio de fragmentos está disponible para BERTopic.'};const a=stages.initial,b=stages.refined,ad=new Set(docsFor(a).filter(d=>!d.regional).map(d=>d.uid)),bd=new Set(docsFor(b).filter(d=>!d.regional).map(d=>d.uid)),bm=new Map(b.chunks.map(c=>[c.uid+'|'+c.id,c])),matrix=new Map();let n=0;for(const x of a.chunks){if(x.assignment===null||!ad.has(x.uid)||!bd.has(x.uid))continue;const y=bm.get(x.uid+'|'+x.id);if(!y||y.text!==x.text)continue;const i=String(x.assignment),j=y.assignment===null?'-2':String(y.assignment),key=i+'|'+j;let item=matrix.get(key);if(!item){item={i,j,value:0,n:0,evidence:[]};matrix.set(key,item);}item.value++;item.evidence.push({uid:x.uid,doc:x.doc,chunk:x.id,value:1});n++;}const left=[...a.layers.bt.topics.map(t=>({id:t.id,name:topicName('initial',t.id)})),{id:'-1',name:'Atípico inicial'}],right=[...b.layers.bt.topics.map(t=>({id:t.id,name:topicName('refined',t.id)})),{id:'-1',name:'Atípico sustantivo'},{id:'-2',name:'Fuera del nivel sustantivo'}];return {items:[...matrix.values()],left,right,n,note:`${n} fragmentos con texto idéntico y asignación inicial exportada · conteos de la muestra, no del corpus completo.`};}
function currentCorr(){return state.method==='chunks'?chunkCorrespondences():documentCorrespondences();}
function pairKey(i,j){return `${state.method}:${state.layer}:${i}:${j}`;}
function pickPair(i,j){pair={i:String(i),j:String(j),method:state.method,layer:state.layer};renderCorr();}
function renderCorr(){const corr=currentCorr(),svg=$('corr-svg');svg.replaceChildren();$('corr-summary').textContent=corr.note;if(!corr.left.length){$('heatmap').innerHTML='';$('pair-detail').innerHTML='';return;}const yLeft=(i)=>70+i*(400/Math.max(1,corr.left.length-1)),yRight=(i)=>45+i*(450/Math.max(1,corr.right.length-1)),li=new Map(corr.left.map((x,i)=>[x.id,i])),ri=new Map(corr.right.map((x,i)=>[x.id,i])),max=Math.max(1,...corr.items.map(e=>e.value));
 for(const e of corr.items){const ann=annotations.pairs[pairKey(e.i,e.j)],show=state.method==='chunks'?e.value>0:e.value>=state.threshold;if(!show&&ann?.status!=='accepted')continue;const x1=265,x2=650,y1=yLeft(li.get(e.i)),y2=yRight(ri.get(e.j));const path=svgNode('path',{d:`M${x1},${y1}C430,${y1} 485,${y2} ${x2},${y2}`,fill:'none',stroke:ann?.status==='rejected'?'#b35454':ann?.status==='accepted'?'#397047':'#96a3ac','stroke-opacity':ann?.status==='rejected'?.3:.3+.45*(e.value/max),'stroke-width':state.method==='chunks'?1+9*Math.sqrt(e.value/max):1+5*e.value,'stroke-dasharray':ann?.status==='rejected'?'5 5':null,cursor:'pointer'});path.append(svgNode('title',{},[`${e.i} → ${e.j}: ${state.method==='chunks'?e.value:fmt(e.value,3)}`]));path.onclick=()=>pickPair(e.i,e.j);svg.append(path);}
 for(const[side,list,yf,x,anchor]of [['initial',corr.left,yLeft,255,'end'],['refined',corr.right,yRight,660,'start']]){list.forEach((t,i)=>{const y=yf(i),g=svgNode('g',{cursor:'pointer'});g.append(svgNode('circle',{cx:x,cy:y,r:7,fill:side==='initial'?'#657b88':'#812b48'}));label(g,x+(side==='initial'?-15:15),y+4,(Number(t.id)>=0?'T'+t.id+' · ':'')+t.name.slice(0,31),{'text-anchor':anchor,'font-size':14});g.onclick=()=>{if(Number(t.id)>=0)selectTopic(side,t.id);};svg.append(g);});}label(svg,250,19,'Inicial',{'text-anchor':'end','font-size':15,'font-weight':700});label(svg,665,19,'Refinada',{'font-size':15,'font-weight':700});
 const lookup=new Map(corr.items.map(e=>[e.i+'|'+e.j,e]));$('heatmap').innerHTML=`<table class="heatmap"><thead><tr><th>Inicial / Refinada</th>${corr.right.map(t=>`<th title="${esc(t.name)}">${Number(t.id)>=0?'T'+t.id:esc(t.name)}</th>`).join('')}</tr></thead><tbody>${corr.left.map(a=>`<tr><th title="${esc(a.name)}">${Number(a.id)>=0?'T'+a.id:esc(a.name)}</th>${corr.right.map(b=>{const e=lookup.get(a.id+'|'+b.id),v=e?.value||0,alpha=state.method==='chunks'?.06+.5*v/max:.05+.55*Math.max(0,v-.45)/.55;return `<td data-pair-i="${a.id}" data-pair-j="${b.id}" class="${pair?.i===a.id&&pair?.j===b.id?'picked':''}" style="background:rgba(129,43,72,${alpha});color:${alpha>.35?'white':'#273640'}">${state.method==='chunks'?v:e?fmt(v,2):'—'}</td>`;}).join('')}</tr>`).join('')}</tbody></table>`;$('heatmap').querySelectorAll('[data-pair-i]').forEach(c=>c.onclick=()=>pickPair(c.dataset.pairI,c.dataset.pairJ));renderPair(corr);}
function renderPair(corr){const box=$('pair-detail');if(!pair||pair.method!==state.method||pair.layer!==state.layer){box.innerHTML='<p class="muted small">Seleccione una relación para consultar los documentos que contribuyen y registrar una decisión experta.</p>';return;}const e=corr.items.find(e=>e.i===pair.i&&e.j===pair.j),key=pairKey(pair.i,pair.j),ann=annotations.pairs[key]||{},left=corr.left.find(t=>t.id===pair.i),right=corr.right.find(t=>t.id===pair.j);if(!left||!right){pair=null;box.innerHTML='';return;}const candidates=corr.items.filter(e=>e.i===pair.i).sort((a,b)=>b.value-a.value),margin=candidates.length>1?candidates[0].value-candidates[1].value:null;const evidence=[...(e?.evidence||[])].slice(0,5);box.innerHTML=`<h3>${esc(left.name)} → ${esc(right.name)}</h3><p class="small">${state.method==='chunks'?'Fragmentos observados: '+(e?.value||0):'Similitud documental: '+(e?fmt(e.value,3):'sin evidencia')}${state.method==='documents'&&margin!==null?'<br>Margen entre las dos mejores correspondencias del tópico inicial: '+fmt(margin,3):''}</p>${state.method==='documents'?'<p class="small muted">Un margen pequeño señala poca discriminación entre alternativas. No es una probabilidad de equivalencia.</p>':''}<h3>Evidencia que contribuye</h3>${evidence.map(r=>`<button class="docrow" data-evidence-doc="${esc(r.doc)}"><strong>${esc(docMaps.initial[r.doc]?.title||r.doc)}</strong><span>${r.chunk?esc(r.chunk):'contribución relativa '+fmt(r.value,4)}</span></button>`).join('')||'<p class="small muted">Sin evidencia exportada para esta relación.</p>'}<h3>Revisión experta</h3><select id="pair-status"><option value="auto">Sin revisión</option><option value="accepted">Aceptar asociación</option><option value="rejected">Rechazar asociación</option></select><textarea id="pair-note" rows="3" maxlength="2000" placeholder="Fundamento de la revisión">${esc(ann.note||'')}</textarea><button id="save-pair">Registrar revisión</button><p class="small muted">La decisión se guarda como anotación; el valor automático continúa visible.</p>`;$('pair-status').value=ann.status||'auto';box.querySelectorAll('[data-evidence-doc]').forEach(b=>b.onclick=()=>selectDoc('initial',b.dataset.evidenceDoc));$('save-pair').onclick=()=>{const before=annotations.pairs[key]||null,after={status:$('pair-status').value,note:$('pair-note').value.trim()};if(after.status==='auto'&&!after.note)delete annotations.pairs[key];else annotations.pairs[key]=after;logEdit('correspondence',key,before,after);renderCorr();toast('Revisión registrada. Guardá la sesión para conservarla.');};}
function renderAudit(){const rows=DATA.coverage.filter(r=>{const d=uidMaps.refined[r.uid]||uidMaps.initial[r.uid];return matches(d,false,{ignoreExclusions:true});}),uids=new Set(rows.map(r=>r.uid)),lines=DATA.lineage.filter(r=>uids.has(r.uid)),recovered=rows.filter(r=>!r.initial_fulltext&&r.refined_fulltext),same=lines.filter(r=>r.status==='conservado').length,added=lines.filter(r=>r.status==='incorporado').length,changed=lines.filter(r=>r.status==='modificado').length;$('audit-metrics').innerHTML=`<div><strong>${recovered.length}</strong><span>textos completos recuperados</span></div><div><strong>${same.toLocaleString('es-AR')}</strong><span>fragmentos con texto conservado</span></div><div><strong>${added}</strong><span>fragmentos incorporados</span></div><div><strong>${changed}</strong><span>fragmentos con texto modificado</span></div>`;$('audit-table').innerHTML=`<table><thead><tr><th>Documento</th><th>Provincia / fuente</th><th>Inicial</th><th>Refinada</th><th>Consulta</th></tr></thead><tbody>${rows.map(r=>`<tr><td>${esc(r.title)}</td><td>${esc(r.regional?'Regional':r.province)}<br>${esc(r.source)}</td><td>${esc(r.initial_origin||'no presente')}</td><td class="audit-status ${!r.initial_fulltext&&r.refined_fulltext?'recovered':''}">${esc(r.refined_origin||'no presente')}${!r.initial_fulltext&&r.refined_fulltext?'<br>Texto completo recuperado':''}</td><td>${r.initial_doc?`<button class="small" data-audit-stage="initial" data-audit-doc="${esc(r.initial_doc)}">Inicial</button>`:''} ${r.refined_doc?`<button class="small" data-audit-stage="refined" data-audit-doc="${esc(r.refined_doc)}">Refinada</button>`:''}</td></tr>`).join('')}</tbody></table>`;const ini=stages.initial,ref=stages.refined,manifest=ref.manifest.source_manifest||{},initialRegionalTraining=ini.chunks.filter(c=>c.assignment!==null&&docMaps.initial[c.doc].regional).length,eligible=ref.chunks.filter(c=>c.assignment!==null),outliers=eligible.filter(c=>c.assignment===-1).length,articleRefs=ref.chunks.filter(c=>c.article).length;const processRows=[['Textos completos SAIJ',ini.documents.filter(d=>d.source==='SAIJ'&&d.full).length+' / '+ini.documents.filter(d=>d.source==='SAIJ').length,ref.documents.filter(d=>d.source==='SAIJ'&&d.full).length+' / '+ref.documents.filter(d=>d.source==='SAIJ').length],['Tópicos BERTopic globales',String(ini.manifest.bertopic_topics),String(manifest.bertopic_topics)],['Nivel sustantivo adicional','No exportado',String(manifest.substantive_level2_topics)+' microtópicos'],['ATACALAR en entrenamiento',initialRegionalTraining+' fragmentos en la muestra exportada',manifest.atacalar_training_excluded?'Excluido del entrenamiento':'No declarado'],['Referencias de artículo estructuradas','Campo no exportado',articleRefs+' fragmentos'],['Atípicos del modelo global',pct(ini.manifest.bertopic_global_outlier_rate),pct(manifest.bertopic_global_outlier_rate)],['Atípicos del nivel sustantivo','No exportado',outliers+' / '+eligible.length+' ('+pct(outliers/eligible.length)+')']];$('processing-table').innerHTML='<p class="small muted">Datos globales de las exportaciones. Cada tasa corresponde al alcance de su propio ajuste.</p><table><thead><tr><th>Dimensión</th><th>Inicial</th><th>Refinada</th></tr></thead><tbody>'+processRows.map(r=>'<tr>'+r.map(v=>'<td>'+esc(v)+'</td>').join('')+'</tr>').join('')+'</tbody></table>';$('audit-table').querySelectorAll('[data-audit-doc]').forEach(b=>b.onclick=()=>{selectDoc(b.dataset.auditStage,b.dataset.auditDoc);$('detail').scrollIntoView({behavior:'smooth',block:'start'});});}
function logEdit(type,key,before,after){annotations.log.push({time:new Date().toISOString(),type,key,before,after});}
function syncControls(){for(const id of ['universe','province','source','period','layer','query','balance'])$(id).value=state[id];$('detail-level').value=state.detail;$('threshold').value=state.threshold;$('threshold-value').textContent=fmt(state.threshold,2);$('density').checked=state.density;$('regional').checked=state.regional;$('correspondence-method').value=state.method;document.querySelectorAll('[data-view]').forEach(b=>b.classList.toggle('active',b.dataset.view===state.view));}
function renderAll(){const panels=$('panels');panels.replaceChildren();panels.classList.toggle('compare',state.view==='compare');for(const sid of state.view==='compare'?['initial','refined']:[state.view])panels.append(renderPanorama(stages[sid]));renderDetail();renderCorr();renderAudit();}
function download(name,body,mime='application/json'){const blob=body instanceof Blob?body:new Blob([body],{type:mime}),url=URL.createObjectURL(blob),a=document.createElement('a');a.href=url;a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(url),10000);}
function csv(rows){if(!rows.length)return '';const cols=Object.keys(rows[0]),cell=x=>'"'+String(x===null||x===undefined?'':typeof x==='object'?JSON.stringify(x):x).replace(/"/g,'""')+'"';return '\ufeff'+[cols.map(cell).join(','),...rows.map(r=>cols.map(k=>cell(r[k])).join(','))].join('\r\n');}
function activeStage(){return state.view==='initial'?'initial':state.view==='refined'?'refined':selection?.stage||'refined';}
function exportedSVG(sid){const original=$('panorama-'+sid);if(!original)throw new Error('Seleccione una etapa visible para exportar.');const clone=original.cloneNode(true);clone.setAttributeNS('http://www.w3.org/2000/xmlns/','xmlns',NS);clone.setAttribute('width','1520');clone.setAttribute('height','1620');clone.setAttribute('viewBox','0 -55 760 810');clone.style.background='white';const rect=svgNode('rect',{x:0,y:-55,width:760,height:810,fill:'white'});clone.prepend(rect);label(clone,30,-25,`Observatorio NOA · ${stages[sid].name} · ${stages[sid].layers[state.layer]?.name||state.layer}`,{'font-size':16,'font-weight':700});label(clone,30,711,`Universo: ${state.universe==='common'?'común':'disponible'} · Fuente: ${state.source} · Provincia: ${state.province}`,{'font-size':11});label(clone,30,729,'Composición provincial y afinidades documentales. Etapas de procesamiento; datos 2009–2019.',{'font-size':10});clone.querySelectorAll('text').forEach(n=>n.setAttribute('font-family','Arial,sans-serif'));clone.querySelectorAll('.topic-label').forEach(n=>{n.setAttribute('font-size','12');n.setAttribute('paint-order','stroke');n.setAttribute('stroke','white');n.setAttribute('stroke-width','4');});clone.querySelectorAll('.province-label').forEach(n=>{n.setAttribute('font-size','19');n.setAttribute('font-weight','600');});clone.querySelectorAll('.edge').forEach(n=>{n.setAttribute('stroke','#8c969e');n.setAttribute('stroke-opacity','.4');});return new XMLSerializer().serializeToString(clone);}
async function exportPNG(){const sid=activeStage();try{const source=exportedSVG(sid),url=URL.createObjectURL(new Blob([source],{type:'image/svg+xml'})),image=new Image();await new Promise((resolve,reject)=>{image.onload=resolve;image.onerror=reject;image.src=url;});const c=document.createElement('canvas');c.width=2280;c.height=2430;c.getContext('2d').drawImage(image,0,0,c.width,c.height);URL.revokeObjectURL(url);c.toBlob(blob=>download(`panorama_${sid}_${state.layer}.png`,blob),'image/png');}catch(e){toast('No se pudo exportar PNG: '+e.message);}}
function saveSession(){download('observatorio_noa_sesion.json',JSON.stringify({schema:'noa-panorama-session/0.1',fingerprints,state,selection,annotations},null,2));}
function importSession(obj){if(obj.schema!=='noa-panorama-session/0.1'||JSON.stringify(obj.fingerprints)!==JSON.stringify(fingerprints))throw new Error('La sesión corresponde a otras exportaciones o tiene un formato no compatible.');const imported={...defaults};for(const k of Object.keys(defaults))if(typeof obj.state?.[k]===typeof defaults[k])imported[k]=obj.state[k];if(!['initial','refined','compare'].includes(imported.view)||!['bt','lda2','lda3'].includes(imported.layer)||!['documents','chunks'].includes(imported.method))throw new Error('Controles de sesión no válidos.');imported.threshold=Math.max(0,Math.min(1,imported.threshold));imported.zoom=Math.max(.65,Math.min(3,imported.zoom));state=imported;const cleanTopics={};for(const[k,v]of Object.entries(obj.annotations?.topics||{})){if(!/^(initial|refined):(bt|lda[23]):\d+$/.test(k)||typeof v!=='object')continue;cleanTopics[k]={label:String(v.label||'').slice(0,90),group:String(v.group||'').slice(0,80),note:String(v.note||'').slice(0,2000)};}const cleanPairs={};for(const[k,v]of Object.entries(obj.annotations?.pairs||{})){if(!/^(documents|chunks):(bt|lda[23]):-?\d+:-?\d+$/.test(k)||!['auto','accepted','rejected'].includes(v?.status))continue;cleanPairs[k]={status:v.status,note:String(v.note||'').slice(0,2000)};}annotations={topics:cleanTopics,pairs:cleanPairs,log:Array.isArray(obj.annotations?.log)?obj.annotations.log.slice(-5000):[]};selection=obj.selection&&['initial','refined'].includes(obj.selection.stage)&&['topic','doc','chunk'].includes(obj.selection.kind)?obj.selection:null;pair=null;syncControls();renderAll();}
for(const id of ['universe','province','source','period','layer','balance'])$(id).addEventListener('change',()=>{state[id]=$(id).value;if(id==='layer'){selection=null;pair=null;}if(id==='province'&&state.province!=='all')state.analysisProvince=state.province;chunkPage=0;renderAll();});let searchTimer;$('query').addEventListener('input',()=>{clearTimeout(searchTimer);searchTimer=setTimeout(()=>{state.query=$('query').value;renderAll();},180);});$('detail-level').onchange=()=>{state.detail=$('detail-level').value;renderAll();};$('threshold').oninput=()=>{state.threshold=Number($('threshold').value);$('threshold-value').textContent=fmt(state.threshold,2);renderAll();};for(const id of ['density','regional'])$(id).onchange=()=>{state[id]=$(id).checked;renderAll();};document.querySelectorAll('[data-view]').forEach(b=>b.onclick=()=>{state.view=b.dataset.view;syncControls();renderAll();});$('reset').onclick=()=>{state={...defaults};selection=null;pair=null;syncControls();renderAll();};$('zoom-in').onclick=()=>{state.zoom=Math.min(3,state.zoom*1.2);renderAll();};$('zoom-out').onclick=()=>{state.zoom=Math.max(.65,state.zoom/1.2);renderAll();};$('zoom-reset').onclick=()=>{state.zoom=1;state.panX=0;state.panY=0;renderAll();};$('correspondence-method').onchange=()=>{state.method=$('correspondence-method').value;pair=null;renderCorr();};$('save-svg').onclick=()=>{try{const sid=activeStage();download(`panorama_${sid}_${state.layer}.svg`,exportedSVG(sid),'image/svg+xml');}catch(e){toast(e.message);}};$('save-png').onclick=exportPNG;$('save-session').onclick=saveSession;$('load-session').onclick=()=>$('session-file').click();$('session-file').onchange=async ev=>{try{const file=ev.target.files[0];if(!file)return;if(file.size>2e6)throw new Error('La sesión supera el tamaño esperado.');importSession(JSON.parse(await file.text()));toast('Sesión cargada');}catch(e){toast(e.message);}ev.target.value='';};$('export-docs').onclick=()=>download('auditoria_documentos.csv',csv(DATA.coverage),'text/csv;charset=utf-8');$('export-chunks').onclick=()=>download('auditoria_fragmentos.csv',csv(DATA.lineage),'text/csv;charset=utf-8');$('export-corr').onclick=()=>{const corr=currentCorr();download('correspondencias_'+state.method+'_'+state.layer+'.csv',csv(corr.items.map(e=>({initial_topic:e.i,refined_topic:e.j,value:e.value,criterion:state.method,layer:state.layer,common_documents:state.method==='documents'?e.n:null,sample_fragments:state.method==='chunks'?corr.n:null,expert_status:annotations.pairs[pairKey(e.i,e.j)]?.status||'auto',expert_note:annotations.pairs[pairKey(e.i,e.j)]?.note||''}))),'text/csv;charset=utf-8');};$('export-log').onclick=()=>download('revisiones_expertas.json',JSON.stringify(annotations,null,2));
const periods=[...new Set(DATA.stages.flatMap(s=>s.documents.map(d=>d.period)).filter(Boolean))].sort();for(const p of periods){const o=document.createElement('option');o.value=p;o.textContent=p;$('period').append(o);}$('limitations').innerHTML=DATA.limitations.map(v=>'<li>'+esc(v)+'</li>').join('');$('provenance').textContent='Exportaciones: '+DATA.stages.map(s=>s.export.filename+' · SHA-256 '+(s.export.sha256?.slice(0,16)||'carpeta local')).join(' | ')+'. Generado: '+DATA.created_at.slice(0,10)+'.';
const ADVANCED=(()=>{
 const newDefaults={excluded:[],analysisTab:'profiles',analysisStage:'refined',analysisProvince:'La Rioja',analysisTopic:'0',liftThreshold:1.2,supportMin:2,labelMode:'keywords'};
 Object.assign(defaults,newDefaults);Object.assign(state,newDefaults);
 const baseSync=syncControls,baseRender=renderAll,baseDetail=renderDetail,baseTopic=selectTopic,baseTopicName=topicName,baseImport=importSession;
 const knownUIDs=new Set(DATA.coverage.map(r=>r.uid));
 const sourceCounts=docs=>({CFI:docs.filter(d=>d.source==='CFI').length,SAIJ:docs.filter(d=>d.source==='SAIJ').length});
 function cleanState(){if(!Array.isArray(state.excluded))state.excluded=[];state.excluded=[...new Set(state.excluded.filter(uid=>knownUIDs.has(uid)))];if(!['profiles','influence','specificity'].includes(state.analysisTab))state.analysisTab='profiles';if(!stages[state.analysisStage])state.analysisStage='refined';if(!DATA.provinces.includes(state.analysisProvince))state.analysisProvince='La Rioja';if(!['keywords','proposed','validated'].includes(state.labelMode))state.labelMode='keywords';state.liftThreshold=Math.max(1,Math.min(3,Number(state.liftThreshold)||1.2));state.supportMin=Math.max(1,Math.min(19,Math.round(Number(state.supportMin)||2)));if(state.source!=='all'&&state.balance==='sources')state.balance='documents';const topics=stages[state.analysisStage].layers[state.layer]?.topics||[];if(!topics.some(t=>t.id===String(state.analysisTopic)))state.analysisTopic=topics[0]?.id||'0';}
 matches=function(doc,full=true,options={}){if(full&&!doc.full)return false;if(!options.ignoreExclusions&&state.excluded.includes(doc.uid))return false;if(doc.regional&&!state.regional)return false;if(!doc.regional&&!doc.compare)return false;if(!options.ignoreProvince&&state.province!=='all'&&(doc.regional||doc.province!==state.province))return false;if(state.source!=='all'&&doc.source!==state.source)return false;if(state.period!=='all'&&doc.period!==state.period)return false;if(state.universe==='common'&&!commonFull.has(doc.uid))return false;if(state.query&&!norm(doc.title+' '+doc.summary+' '+txt(doc.text)).includes(norm(state.query)))return false;return true;};
 function analyticalDocs(stage,options={}){return stage.documents.filter(d=>!d.regional&&d.compare&&matches(d,true,options)&&profile(d,state.layer));}
 function averagingWeights(docs,mode='documents'){if(!docs.length)return null;if(mode==='documents')return docs.map(()=>1/docs.length);const n=sourceCounts(docs);if(!n.CFI||!n.SAIJ)return null;return docs.map(d=>.5/n[d.source]);}
 function average(docs,mode='documents'){const w=averagingWeights(docs,mode);if(!w)return null;const k=profile(docs[0],state.layer).length;return Array.from({length:k},(_,i)=>docs.reduce((a,d,j)=>a+w[j]*profile(d,state.layer)[i],0));}
 function profiles(stage,ignoreProvince=false){const docs=analyticalDocs(stage,{ignoreProvince});return DATA.provinces.map(province=>{const ds=docs.filter(d=>d.province===province),n=sourceCounts(ds);return {province,docs:ds,n:ds.length,cfi:n.CFI,saij:n.SAIJ,observed:average(ds),balanced:average(ds,'sources')};});}
 function activeVector(group){return state.balance==='sources'?group.balanced:group.observed;}
 function contextRows(){return {universe:state.universe,source_filter:state.source,province_filter:state.province,period:state.period,query:state.query,excluded_documents:state.excluded.map(uid=>(uidMaps.refined[uid]||uidMaps.initial[uid]).id).join('|'),profiles_frozen:true,initial_sha256:stages.initial.export.sha256,refined_sha256:stages.refined.export.sha256};}
 function contribution(stage,province,topic){const all=analyticalDocs(stage,{ignoreExclusions:true}).filter(d=>d.province===province),active=all.filter(d=>!state.excluded.includes(d.uid)),p=average(active,state.balance),w=averagingWeights(active,state.balance),index=new Map(active.map((d,i)=>[d.uid,i]));return {all,active,profile:p,rows:all.map(d=>{const i=index.get(d.uid),excluded=i===undefined,rest=active.filter(x=>x.uid!==d.uid),without=excluded?null:average(rest,state.balance),v=profile(d,state.layer)?.[topic]??null;return {stage:stage.id,layer:state.layer,province,topic,mode:state.balance,doc_id:d.id,uid:d.uid,title:d.title,source:d.source,n_active:active.length,excluded,document_affinity:v,document_weight:!excluded&&w?w[i]:null,contribution_share:!excluded&&w&&p?.[topic]>0?w[i]*v/p[topic]:null,mean_current:p?.[topic]??null,mean_without_document:without?.[topic]??null,delta_pp:without&&p?100*(without[topic]-p[topic]):null,leader_before:p?p.indexOf(Math.max(...p)):null,leader_after:without?without.indexOf(Math.max(...without)):null,balance_lost:!excluded&&state.balance==='sources'&&rest.length>0&&!without,profiles_frozen:true};})};}
 function concentration(stage){const groups=profiles(stage,true),available=groups.filter(g=>activeVector(g)),k=stage.layers[state.layer]?.topics.length||0,reference=Array.from({length:k},(_,i)=>available.length?available.reduce((a,g)=>a+activeVector(g)[i],0)/available.length:null),rows=[];for(const g of groups){const p=activeVector(g);for(let i=0;i<k;i++){const support=p?g.docs.filter(d=>{const a=profile(d,state.layer),ordered=[...a].sort((a,b)=>b-a);return a[i]>=ordered[Math.min(1,a.length-1)]-1e-9;}):[];rows.push({stage:stage.id,layer:state.layer,mode:state.balance,province:g.province,topic:i,n_documents:g.n,n_cfi:g.cfi,n_saij:g.saij,affinity:p?.[i]??null,reference:reference[i],reference_provinces:available.length,relative_concentration:p&&available.length>=2&&reference[i]>0?p[i]/reference[i]:null,support_top2:p?support.length:null,support_doc_ids:support.map(d=>d.id).join('|'),support_cfi:p?support.filter(d=>d.source==='CFI').length:null,support_saij:p?support.filter(d=>d.source==='SAIJ').length:null,reference_ignores_province_filter:true});}}return {groups,reference,n:available.length,rows};}
 function proposal(sid,tid,layer=state.layer){return (DATA.label_proposals||[]).find(p=>p.stage===sid&&p.layer===layer&&String(p.topic)===String(tid));}
 topicName=function(sid,tid,layer=state.layer){const custom=annotations.topics[topicKey(sid,layer,tid)],proposed=proposal(sid,tid,layer),topic=stages[sid].layers[layer]?.topics.find(t=>t.id===String(tid));const words=topic?.words.slice(0,2).map(w=>w.term.replace(/_/g,' ')).join(' · ')||'Tópico '+tid;if(state.labelMode==='keywords')return words;if(custom?.status==='validated'&&custom.label)return custom.label;if(state.labelMode==='validated')return words;return custom?.label||proposed?.label||words;};
 selectTopic=function(sid,tid){state.analysisStage=sid;state.analysisTopic=String(tid);if(state.province!=='all')state.analysisProvince=state.province;baseTopic(sid,tid);};
 function setExcluded(uid,exclude=true){if(!knownUIDs.has(uid))throw new Error('Documento no reconocido');const before=state.excluded.includes(uid);state.excluded=exclude?[...new Set([...state.excluded,uid])]:state.excluded.filter(x=>x!==uid);if(before!==exclude)logEdit('scenario_exclusion',uid,before,exclude);syncControls();renderAll();}
 function scenario(){const box=$('scenario-strip');box.innerHTML=state.excluded.length?`<strong>Escenario: ${state.excluded.length} documento(s) excluido(s)</strong>${state.excluded.map(uid=>{const d=uidMaps.refined[uid]||uidMaps.initial[uid];return `<button class="excluded-chip" data-restore="${uid}" title="${esc(d.title)}">Restaurar ${esc(d.id)}</button>`;}).join('')}<button id="restore-all">Restaurar todos</button>`:'<span>Escenario completo: todos los documentos que cumplen los filtros.</span><span class="muted">Las exclusiones son reversibles y se aplican a ambas etapas.</span>';box.querySelectorAll('[data-restore]').forEach(b=>b.onclick=()=>setExcluded(b.dataset.restore,false));if($('restore-all'))$('restore-all').onclick=()=>{logEdit('scenario_restore_all','all',[...state.excluded],[]);state.excluded=[];renderAll();};}
 function displayGroups(groups){return state.province==='all'?groups:groups.filter(g=>g.province===state.province);}
 function renderProfiles(stage){const groups=displayGroups(profiles(stage));return `<p class="analytic-note">Ambos perfiles usan los mismos documentos de la etapa y el escenario. El perfil equilibrado otorga 50 % a cada fuente cuando CFI y SAIJ están presentes.</p><div class="profiles-grid">${groups.map(g=>`<article class="profile-card" data-profile-province="${esc(g.province)}"><h3 style="color:${COLORS[g.province]}">${esc(g.province)}</h3><div class="small muted">${g.n} documentos · ${g.cfi} CFI / ${g.saij} SAIJ</div><div class="profile-legend"><span>● Observado</span><span style="color:#812b48">● Equilibrado</span></div><div class="profile-row profile-head"><span>Tópico</span><span>Dos ponderaciones</span><span>Obs.</span><span>Equil.</span></div>${(stage.layers[state.layer]?.topics||[]).map((t,i)=>`<div class="profile-row" data-profile-topic="${t.id}"><button data-analytic-topic="${t.id}" title="${esc(topicName(stage.id,t.id))}">T${t.id}</button><div class="profile-track"><i class="observed" style="width:${100*(g.observed?.[i]||0)}%"></i>${g.balanced?`<i class="balanced" style="width:${100*g.balanced[i]}%"></i>`:''}</div><span data-observed="${g.observed?.[i]??''}">${g.observed?pct(g.observed[i]):'ND'}</span><span data-balanced="${g.balanced?.[i]??''}">${g.balanced?pct(g.balanced[i]):'ND'}</span></div>`).join('')}${!g.balanced?'<div class="notice">Equilibrio CFI/SAIJ no disponible: falta evidencia de una fuente.</div>':`<p class="help-inline">Primer tópico observado: T${g.observed.indexOf(Math.max(...g.observed))}. Equilibrado: T${g.balanced.indexOf(Math.max(...g.balanced))}.</p>`}</article>`).join('')}</div>`;}
 function renderInfluence(stage){const topic=Number(state.analysisTopic),result=contribution(stage,state.analysisProvince,topic),p=result.profile,valid=result.rows.filter(r=>r.contribution_share!==null),largest=valid.length?Math.max(...valid.map(r=>r.contribution_share)):null;return `<p class="analytic-note">${esc(state.analysisProvince)} · T${topic}: ${esc(topicName(stage.id,topic))} · ${state.balance==='sources'?'CFI y SAIJ equilibrados':'igual peso por documento'}. La exclusión de un documento se aplica a las dos etapas por su identidad.</p><div class="influence-summary"><div><strong>${p?pct(p[topic]):'ND'}</strong><span>afinidad provincial actual</span></div><div><strong>${result.active.length}</strong><span>documentos activos</span></div><div><strong>${largest!==null?pct(largest):'ND'}</strong><span>mayor aporte de un documento</span></div></div>${!p?'<div class="notice">Perfil no disponible para esta ponderación. Restaure documentos, ajuste los filtros o seleccione el perfil observado.</div>':''}<div class="table-wrap"><table class="influence-table"><thead><tr><th>Escenario</th><th>Documento</th><th>Aporte al tópico</th><th>Afinidad documental</th><th>Perfil sin este documento</th><th>Cambio</th></tr></thead><tbody>${[...result.rows].sort((a,b)=>(b.contribution_share??-1)-(a.contribution_share??-1)).map(r=>`<tr class="${r.excluded?'excluded-row':''}" data-influence-doc="${r.doc_id}"><td><label class="check"><input type="checkbox" data-exclude="${r.uid}" ${r.excluded?'checked':''}> Excluir</label></td><td class="document-title"><button data-influence-open="${r.doc_id}">${esc(r.doc_id)} · ${esc(r.title)}</button><div class="muted">${esc(r.source)}${r.excluded?' · excluido del escenario':''}</div></td><td>${r.contribution_share!==null?`<span class="support-bar"><i style="width:${100*r.contribution_share}%"></i></span>${pct(r.contribution_share)}`:'—'}</td><td>${r.document_affinity!==null?pct(r.document_affinity):'ND'}</td><td>${r.mean_without_document!==null?pct(r.mean_without_document):r.balance_lost?'ND: pierde balance de fuentes':'—'}</td><td>${r.delta_pp!==null?(r.delta_pp>=0?'+':'')+fmt(r.delta_pp,2)+' pp':'—'}${r.leader_after!==null?`<br><span class="muted">Primero: T${r.leader_before} → T${r.leader_after}</span>`:''}</td></tr>`).join('')}</tbody></table></div><p class="help-inline">El aporte divide la contribución ponderada de cada documento por la afinidad provincial del tópico; suma 100 % entre documentos activos. La afinidad documental y el aporte son medidas diferentes. Retirar documentos recalcula el promedio con los perfiles preservados.</p>`;}
 function liftColor(value){if(value===null)return '#f0f1f2';const ratio=Math.max(0,Math.min(2.5,value));return ratio>=1?`rgba(185,133,56,${.08+.55*Math.min(1,(ratio-1)/1.5)})`:`rgba(40,120,168,${.08+.4*(1-ratio)})`;}
 function renderConcentration(stage){const result=concentration(stage),rows=result.rows.filter(r=>state.province==='all'||r.province===state.province),topics=stage.layers[state.layer]?.topics||[];return `<p class="analytic-note">Concentración relativa = afinidad provincial / promedio de las provincias disponibles, cada provincia con el mismo peso. Un valor 1 equivale a ese promedio; 1,5 indica una afinidad 50 % mayor. El indicador describe presencia relativa.</p><p class="reference-line">Referencia: ${result.n} provincias disponibles con los mismos filtros de fuente, período, búsqueda, universo y exclusiones. El filtro provincial selecciona filas y conserva la referencia comparativa.</p>${result.n<2?'<div class="notice">Se requieren al menos dos provincias para interpretar una concentración comparativa.</div>':''}<div class="table-wrap"><table class="heatmap"><thead><tr><th>Provincia / tópico</th>${topics.map(t=>`<th title="${esc(topicName(stage.id,t.id))}">T${t.id}</th>`).join('')}</tr></thead><tbody>${DATA.provinces.filter(p=>state.province==='all'||p===state.province).map(p=>`<tr><th>${esc(p)}</th>${topics.map(t=>{const r=rows.find(r=>r.province===p&&r.topic===Number(t.id)),v=r?.relative_concentration??null,mark=v!==null&&v>=state.liftThreshold&&r.support_top2>=state.supportMin;return `<td class="lift-cell ${mark?'lift-mark':''}" data-lift-province="${esc(p)}" data-lift-topic="${t.id}" style="background:${liftColor(v)}" title="Afinidad ${r?.affinity!==null?pct(r.affinity):'ND'}; soporte en los dos mayores tópicos: ${r?.support_doc_ids||'sin soporte'}"><strong>${v===null?'ND':fmt(v,2)+'×'}</strong><span>${r?.support_top2===null?'sin balance':(r?.support_top2||0)+' / '+(r?.n_documents||0)+' docs.'}</span></td>`;}).join('')}</tr>`).join('')}</tbody></table></div><p class="help-inline">La cifra inferior cuenta documentos donde el tópico se encuentra entre las dos mayores afinidades, incluidos empates. En K=2 todos los documentos cumplen ese criterio. El borde destaca valores ≥ ${fmt(state.liftThreshold,2)}× y soporte en al menos ${state.supportMin} documentos; estos umbrales son exploratorios y editables.</p>`;}
 function renderAnalytics(){const stage=stages[state.analysisStage];$('analysis-stage').value=state.analysisStage;$('analysis-province').value=state.analysisProvince;const topics=stage.layers[state.layer]?.topics||[];$('analysis-topic').innerHTML=topics.map(t=>`<option value="${t.id}">T${t.id} · ${esc(topicName(stage.id,t.id).slice(0,48))}</option>`).join('');$('analysis-topic').value=state.analysisTopic;$('lift-threshold').value=state.liftThreshold;$('support-min').value=state.supportMin;document.querySelectorAll('[data-analysis-tab]').forEach(b=>b.classList.toggle('active',b.dataset.analysisTab===state.analysisTab));$('analysis-province-control').classList.toggle('hidden',state.analysisTab!=='influence');$('analysis-topic-control').classList.toggle('hidden',state.analysisTab!=='influence');$('lift-threshold-control').classList.toggle('hidden',state.analysisTab!=='specificity');$('support-min-control').classList.toggle('hidden',state.analysisTab!=='specificity');const box=$('analysis-content');box.innerHTML=!topics.length?'<div class="notice">Esta capa no está exportada en la etapa seleccionada.</div>':state.analysisTab==='profiles'?renderProfiles(stage):state.analysisTab==='influence'?renderInfluence(stage):renderConcentration(stage);box.querySelectorAll('[data-analytic-topic]').forEach(b=>b.onclick=()=>selectTopic(stage.id,b.dataset.analyticTopic));box.querySelectorAll('[data-exclude]').forEach(b=>b.onchange=()=>setExcluded(b.dataset.exclude,b.checked));box.querySelectorAll('[data-influence-open]').forEach(b=>b.onclick=()=>selectDoc(stage.id,b.dataset.influenceOpen));box.querySelectorAll('[data-lift-topic]').forEach(b=>b.onclick=()=>{state.analysisProvince=b.dataset.liftProvince;state.analysisTopic=b.dataset.liftTopic;state.analysisTab='influence';selectTopic(stage.id,b.dataset.liftTopic);});}
 renderDetail=function(){baseDetail();const stage=selection?stages[selection.stage]:null;if(!stage)return;const box=$('detail');if(selection.kind==='topic'){const tid=selection.id,key=topicKey(stage.id,selection.layer,tid),p=proposal(stage.id,tid,selection.layer),custom=annotations.topics[key]||{};if(p){const section=document.createElement('div');section.className='interpretation';section.innerHTML=`<h3>Lectura interpretativa propuesta</h3><strong>${esc(p.label)}</strong><p class="small">${esc(p.description)}</p><p class="proposed-label">Pendiente de validación experta</p>${p.evidence_refs.map(ref=>`<button class="review-ref" data-proposal-ref="${esc(ref)}">${esc(ref)}</button>`).join('')}<button id="use-proposal" class="small">Usar como borrador de etiqueta</button>`;box.querySelector('h2').after(section);section.querySelectorAll('[data-proposal-ref]').forEach(b=>b.onclick=()=>selectChunk(stage.id,b.dataset.proposalRef));$('use-proposal').onclick=()=>{$('topic-label').value=p.label;$('topic-refs').value=p.evidence_refs.join('\n');$('topic-note').value=p.description;$('topic-status').value='proposed';box.querySelector('details:last-of-type').open=true;};}
 const form=$('topic-note')?.closest('details');if(form){const area=document.createElement('div');area.innerHTML=`<label class="small">Estado de revisión<select id="topic-status" style="display:block;width:100%"><option value="proposed">Propuesta / pendiente</option><option value="validated">Validada por el investigador</option></select></label><label class="small">Referencias documentales o de fragmentos<textarea id="topic-refs" rows="2" placeholder="DOC_007_CH_0003, DOC_011_CH_0002">${esc((custom.evidence_refs||[]).join('\n'))}</textarea></label>`;form.insertBefore(area,$('save-topic'));$('topic-status').value=custom.status||'proposed';$('save-topic').onclick=()=>{const labelValue=$('topic-label').value.trim(),noteValue=$('topic-note').value.trim(),refs=$('topic-refs').value.split(/[,;\n]+/).map(s=>s.trim()).filter(Boolean),status=$('topic-status').value,validRefs=new Set([...stage.documents.map(d=>d.id),...stage.chunks.map(c=>c.id)]);if(refs.some(r=>!validRefs.has(r))){toast('Revise las referencias: deben existir en esta exportación.');return;}if(status==='validated'&&(!labelValue||!noteValue||!refs.length)){toast('Para validar, indique etiqueta, fundamento y al menos una referencia.');return;}const before=annotations.topics[key]||null,after={label:labelValue,group:$('topic-group').value.trim(),note:noteValue,status,evidence_refs:refs,reviewed_at:status==='validated'?new Date().toISOString():null};annotations.topics[key]=after;state.labelMode=status==='validated'?'validated':'proposed';logEdit('topic',key,before,after);syncControls();renderAll();toast('Revisión registrada. Guarde la sesión para conservarla.');};}
 const marker=document.createElement('p');marker.className=custom.status==='validated'?'validated-label':'proposed-label';marker.textContent=custom.status==='validated'?'Etiqueta validada con referencias registradas':'Etiqueta interpretativa pendiente de validación';box.querySelector('h2').after(marker);const button=document.createElement('button');button.textContent='Ver aporte documental y exclusión';button.style.marginTop='8px';button.onclick=()=>{state.analysisStage=stage.id;state.analysisTopic=tid;state.analysisTab='influence';renderAnalytics();$('provincial-analysis').scrollIntoView({behavior:'smooth'});};box.append(button);
 }else{const d=docMaps[stage.id][selection.kind==='chunk'?selection.doc:selection.id];if(!d)return;const block=document.createElement('div');block.innerHTML=`<button id="toggle-selected-doc" class="small">${state.excluded.includes(d.uid)?'Restaurar en el escenario':'Excluir temporalmente del escenario'}</button>${state.excluded.includes(d.uid)?'<p class="notice">Documento excluido del escenario; su evidencia permanece consultable.</p>':''}`;box.querySelector('h2').after(block);$('toggle-selected-doc').onclick=()=>setExcluded(d.uid,!state.excluded.includes(d.uid));for(const review of DATA.catalogue_reviews||[])if(review.doc_id===d.id){const note=document.createElement('div');note.className='notice';note.textContent=review.observation;box.append(note);}}};
 documentCorrespondences=function(){const a=stages.initial,b=stages.refined,la=a.layers[state.layer],lb=b.layers[state.layer];if(!la||!lb)return {items:[],left:[],right:[],n:0,note:'Capa no exportada en ambas etapas.'};const bm=new Map(analyticalDocs(b).map(d=>[d.uid,d]));let ds=analyticalDocs(a).filter(d=>bm.has(d.uid));if(state.balance==='sources')ds=ds.filter(d=>{const same=ds.filter(x=>x.province===d.province),counts=sourceCounts(same);return counts.CFI&&counts.SAIJ;});const w=docWeights(ds),items=[];for(let i=0;i<la.topics.length;i++)for(let j=0;j<lb.topics.length;j++){const av=ds.map(d=>profile(d,state.layer)[i]),bv=ds.map(d=>profile(bm.get(d.uid),state.layer)[j]),value=cosine(av,bv,w);if(value!==null)items.push({i:String(i),j:String(j),value,n:ds.length,evidence:ds.map((d,k)=>({uid:d.uid,doc:d.id,value:av[k]*bv[k]})).sort((x,y)=>y.value-x.value)});}return {items,left:la.topics.map(t=>({id:t.id,name:topicName('initial',t.id)})),right:lb.topics.map(t=>({id:t.id,name:topicName('refined',t.id)})),n:ds.length,note:`${ds.length} textos completos comunes${state.balance==='sources'?' con ambas fuentes representadas por provincia':''} · perfiles preservados.`};};
 syncControls=function(){cleanState();baseSync();$('label-mode').value=state.labelMode;$('balance').querySelector('[value=sources]').disabled=state.source!=='all';};
 renderAll=function(){syncControls();baseRender();scenario();renderAnalytics();};
 function tableRows(){const stage=stages[state.analysisStage],context=contextRows();if(state.analysisTab==='influence')return contribution(stage,state.analysisProvince,Number(state.analysisTopic)).rows.map(r=>({...r,...context}));if(state.analysisTab==='specificity')return concentration(stage).rows.filter(r=>state.province==='all'||r.province===state.province).map(r=>({...r,...context}));return displayGroups(profiles(stage)).flatMap(g=>(stage.layers[state.layer]?.topics||[]).map((t,i)=>({stage:stage.id,layer:state.layer,province:g.province,topic:t.id,n_documents:g.n,n_cfi:g.cfi,n_saij:g.saij,observed:g.observed?.[i]??null,balanced:g.balanced?.[i]??null,...context})));}
 function currentFigure(){const stage=stages[state.analysisStage],topics=stage.layers[state.layer]?.topics||[],width=1440,groups=displayGroups(profiles(stage)),result=contribution(stage,state.analysisProvince,Number(state.analysisTopic));let height=state.analysisTab==='profiles'?220+Math.max(1,topics.length)*47:state.analysisTab==='influence'?235+Math.max(1,result.rows.length)*66:390;const svg=svgNode('svg',{viewBox:`0 0 ${width} ${height}`,width,height});svg.setAttributeNS('http://www.w3.org/2000/xmlns/','xmlns',NS);svg.append(svgNode('rect',{x:0,y:0,width,height,fill:'white'}));const title=state.analysisTab==='profiles'?'Perfiles observados y equilibrados':state.analysisTab==='influence'?'Aporte documental y exclusión':'Concentración temática relativa';label(svg,35,40,`Observatorio NOA · ${stage.name} · ${title}`,{'font-size':26,'font-weight':700,'font-family':'Arial'});label(svg,35,69,`${stage.layers[state.layer]?.name||state.layer} · universo ${state.universe==='common'?'común':'disponible'} · fuente ${state.source} · período ${state.period}`,{'font-size':15});
 if(state.analysisTab==='profiles'){const maxValue=Math.max(.1,...groups.flatMap(g=>[...(g.observed||[]),...(g.balanced||[])])),scale=Math.ceil(maxValue*20)/20;groups.forEach((g,index)=>{const x=35+index*(1370/Math.max(1,groups.length));label(svg,x,110,g.province,{'font-size':22,fill:COLORS[g.province],'font-weight':700});label(svg,x,135,`${g.n} documentos · ${g.cfi} CFI / ${g.saij} SAIJ`,{'font-size':14});topics.forEach((t,i)=>{const y=170+i*47;label(svg,x,y+8,'T'+t.id,{'font-size':15});svg.append(svgNode('rect',{x:x+40,y:y-3,width:245,height:21,fill:'#f1f3f4'}));if(g.observed)svg.append(svgNode('rect',{x:x+40,y:y-3,width:245*g.observed[i]/scale,height:8,fill:'#899aa5'}));if(g.balanced)svg.append(svgNode('rect',{x:x+40,y:y+10,width:245*g.balanced[i]/scale,height:8,fill:'#812b48'}));label(svg,x+300,y+7,g.observed?pct(g.observed[i]):'ND',{'font-size':13});label(svg,x+365,y+7,g.balanced?pct(g.balanced[i]):'ND',{'font-size':13,fill:'#812b48'});});});label(svg,35,height-38,`Gris: observado · Bordó: CFI/SAIJ equilibrados · Escala de barras: 0–${pct(scale)}. ND indica evidencia insuficiente para el balance.`,{'font-size':14});}
 else if(state.analysisTab==='influence'){const current=result.profile?.[Number(state.analysisTopic)];label(svg,35,110,`${state.analysisProvince} · T${state.analysisTopic} · Afinidad actual: ${current===undefined?'ND':pct(current)} · ${state.balance==='sources'?'fuentes equilibradas':'igual peso por documento'}`,{'font-size':19});label(svg,660,143,'Aporte al tópico',{'font-size':14});label(svg,1100,143,'Perfil sin documento',{'font-size':14});[...result.rows].sort((a,b)=>(b.contribution_share??-1)-(a.contribution_share??-1)).forEach((r,i)=>{const y=175+i*66;label(svg,35,y,`${r.doc_id} · ${r.source} ${r.excluded?'· excluido':''}`,{'font-size':16,fill:r.excluded?'#999':'#273640'});label(svg,35,y+22,r.title.length>76?r.title.slice(0,75)+'…':r.title,{'font-size':13});if(r.contribution_share!==null){svg.append(svgNode('rect',{x:660,y:y-12,width:340*r.contribution_share,height:18,fill:'#812b48'}));label(svg,1010,y+3,pct(r.contribution_share),{'font-size':15});}label(svg,1100,y+3,r.mean_without_document!==null?pct(r.mean_without_document):'ND',{'font-size':16});label(svg,1230,y+3,r.delta_pp!==null?(r.delta_pp>=0?'+':'')+fmt(r.delta_pp,2)+' pp':'',{'font-size':14});});label(svg,35,height-38,'Aportes normalizados entre documentos activos. Exclusión con perfiles fijos; el balance requiere mantener ambas fuentes.',{'font-size':14});}
 else{const result=concentration(stage),display=DATA.provinces.filter(p=>state.province==='all'||p===state.province),cellWidth=Math.min(125,1090/Math.max(1,topics.length)),start=300;topics.forEach((t,i)=>label(svg,start+i*cellWidth+cellWidth/2,117,'T'+t.id,{'font-size':17,'text-anchor':'middle'}));display.forEach((p,j)=>{const y=145+j*59;label(svg,35,y+30,p,{'font-size':20,fill:COLORS[p]});topics.forEach((t,i)=>{const r=result.rows.find(r=>r.province===p&&r.topic===Number(t.id)),v=r?.relative_concentration??null,marked=v!==null&&v>=state.liftThreshold&&r.support_top2>=state.supportMin;svg.append(svgNode('rect',{x:start+i*cellWidth,y,width:cellWidth-3,height:51,fill:liftColor(v),stroke:marked?'#812b48':'#fff','stroke-width':marked?2:1}));label(svg,start+i*cellWidth+cellWidth/2,y+22,v===null?'ND':fmt(v,2)+'×',{'font-size':17,'text-anchor':'middle'});label(svg,start+i*cellWidth+cellWidth/2,y+41,r?.support_top2===null?'':`${r.support_top2}/${r.n_documents} docs.`,{'font-size':12,'text-anchor':'middle'});});});label(svg,35,height-41,`Referencia: ${result.n} provincias, igual peso por provincia. 1× = promedio. Soporte: tópico entre las dos mayores afinidades documentales.`,{'font-size':14});}
 label(svg,35,height-15,`Escenario: ${state.excluded.length} documentos excluidos · perfiles preservados · datos documentales 2009–2019.`,{'font-size':12,fill:'#65717a'});svg.querySelectorAll('text').forEach(t=>t.setAttribute('font-family','Arial,sans-serif'));return new XMLSerializer().serializeToString(svg);}
 async function analysisPNG(){try{const url=URL.createObjectURL(new Blob([currentFigure()],{type:'image/svg+xml'})),image=new Image();await new Promise((resolve,reject)=>{image.onload=resolve;image.onerror=reject;image.src=url;});const canvas=document.createElement('canvas');canvas.width=image.width*2;canvas.height=image.height*2;canvas.getContext('2d').drawImage(image,0,0,canvas.width,canvas.height);URL.revokeObjectURL(url);canvas.toBlob(b=>download('analisis_'+state.analysisTab+'_'+state.analysisStage+'.png',b),'image/png');}catch(e){toast('No se pudo exportar la figura: '+e.message);}}
 saveSession=function(){download('observatorio_noa_sesion_v02.json',JSON.stringify({schema:'noa-panorama-session/0.2',fingerprints,state,selection,annotations},null,2));};
 importSession=function(obj){if(!['noa-panorama-session/0.1','noa-panorama-session/0.2'].includes(obj.schema))throw new Error('Formato de sesión no compatible');if(JSON.stringify(obj.fingerprints)!==JSON.stringify(fingerprints))throw new Error('La sesión corresponde a otras exportaciones');const imported={...defaults};for(const k of Object.keys(defaults))if(typeof obj.state?.[k]===typeof defaults[k])imported[k]=obj.state[k];if(!['initial','refined','compare'].includes(imported.view)||!['bt','lda2','lda3'].includes(imported.layer)||!['documents','chunks'].includes(imported.method))throw new Error('Controles de sesión no válidos');const cleanTopics={};for(const[k,v]of Object.entries(obj.annotations?.topics||{})){if(!/^(initial|refined):(bt|lda[23]):\d+$/.test(k)||!v||typeof v!=='object')continue;const sid=k.split(':')[0],refs=Array.isArray(v.evidence_refs)?v.evidence_refs.filter(r=>typeof r==='string'&&(docMaps[sid][r]||stages[sid].chunks.some(c=>c.id===r))).slice(0,50):[],labelValue=String(v.label||'').slice(0,90),noteValue=String(v.note||'').slice(0,2000);cleanTopics[k]={label:labelValue,group:String(v.group||'').slice(0,80),note:noteValue,status:v.status==='validated'&&refs.length&&labelValue&&noteValue?'validated':'proposed',evidence_refs:refs,reviewed_at:typeof v.reviewed_at==='string'?v.reviewed_at:null};}const cleanPairs={};for(const[k,v]of Object.entries(obj.annotations?.pairs||{}))if(/^(documents|chunks):(bt|lda[23]):-?\d+:-?\d+$/.test(k)&&['auto','accepted','rejected'].includes(v?.status))cleanPairs[k]={status:v.status,note:String(v.note||'').slice(0,2000)};state=imported;state.threshold=Math.max(0,Math.min(1,state.threshold));state.zoom=Math.max(.65,Math.min(3,state.zoom));annotations={topics:cleanTopics,pairs:cleanPairs,log:Array.isArray(obj.annotations?.log)?obj.annotations.log.slice(-5000):[]};selection=obj.selection&&['initial','refined'].includes(obj.selection.stage)&&['topic','doc','chunk'].includes(obj.selection.kind)?obj.selection:null;pair=null;cleanState();syncControls();renderAll();};
 $('save-session').onclick=saveSession;$('label-mode').onchange=()=>{state.labelMode=$('label-mode').value;renderAll();};document.querySelectorAll('[data-analysis-tab]').forEach(b=>b.onclick=()=>{state.analysisTab=b.dataset.analysisTab;renderAnalytics();});$('analysis-stage').onchange=()=>{state.analysisStage=$('analysis-stage').value;cleanState();renderAnalytics();};$('analysis-province').onchange=()=>{state.analysisProvince=$('analysis-province').value;renderAnalytics();};$('analysis-topic').onchange=()=>{state.analysisTopic=$('analysis-topic').value;renderAnalytics();};$('lift-threshold').onchange=()=>{state.liftThreshold=Number($('lift-threshold').value);cleanState();renderAnalytics();};$('support-min').onchange=()=>{state.supportMin=Number($('support-min').value);cleanState();renderAnalytics();};$('export-analysis-csv').onclick=()=>download('analisis_'+state.analysisTab+'_'+state.analysisStage+'.csv',csv(tableRows()),'text/csv;charset=utf-8');$('export-analysis-svg').onclick=()=>download('analisis_'+state.analysisTab+'_'+state.analysisStage+'.svg',currentFigure(),'image/svg+xml');$('export-analysis-png').onclick=analysisPNG;
 return {getProfiles:(sid='refined',ignoreProvince=false)=>profiles(stages[sid],ignoreProvince),getInfluence:(sid='refined',province='La Rioja',topic=0)=>contribution(stages[sid],province,topic),getConcentration:(sid='refined')=>concentration(stages[sid]),excludeDocument:(id,exclude=true)=>{const d=docMaps.refined[id]||docMaps.initial[id];if(!d)throw new Error('Documento no reconocido');setExcluded(d.uid,exclude);},analysisFigure:currentFigure,analysisRows:tableRows,importSession,selectTopic,render:()=>renderAll()};
})();

window.NOA={DATA,getState:()=>({...state}),getSelection:()=>selection,getAnnotations:()=>annotations,setState:patch=>{state={...state,...patch};syncControls();renderAll();},selectTopic,selectDoc,selectChunk,getCorrespondences:currentCorr,importSession,exportedSVG,render:renderAll};Object.assign(window.NOA,ADVANCED);syncControls();renderAll();
</script>
</body>
</html>

Writing visor_template.html


## 5. Cargar los datos
El ZIP de resultados se lee como datos: se utilizan su JSON y sus referencias. El código del ZIP cargado no se ejecuta. Si elegís originales, se identifican las dos etapas por sus archivos internos.

In [10]:
import importlib
import json
import zipfile
import noa_analysis
import noa_panorama
importlib.reload(noa_analysis)
importlib.reload(noa_panorama)
from noa_analysis import load_saved_results, provincial_profiles, influence, concentration
from noa_panorama import open_export, build_payload, write_project

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
def cargar_desde_colab():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Completá las rutas de entrada para ejecutar fuera de Colab.") from exc
    return list(files.upload())

if MODO_CARGA == "resultados":
    if not RUTA_RESULTADOS:
        print("Seleccioná el ZIP de resultados v0.1/v0.2 o panorama_datos.json.")
        candidatos = []
        for nombre in cargar_desde_colab():
            if Path(nombre).suffix.lower() == ".json":
                candidatos.append(nombre)
            elif Path(nombre).suffix.lower() == ".zip":
                with zipfile.ZipFile(nombre) as archivo:
                    if any(n.rsplit("/", 1)[-1] == "panorama_datos.json" for n in archivo.namelist()):
                        candidatos.append(nombre)
        if len(candidatos) != 1:
            raise ValueError("Seleccioná un único archivo compatible o completá RUTA_RESULTADOS.")
        RUTA_RESULTADOS = candidatos[0]
    datos = load_saved_results(RUTA_RESULTADOS)
else:
    if not RUTA_INICIAL or not RUTA_REFINADA:
        print("Seleccioná outputs_1821 y observatorio_noa_rag_v302 en la misma carga.")
        for nombre in cargar_desde_colab():
            if Path(nombre).suffix.lower() not in [".rar", ".zip"]:
                continue
            raiz, _ = open_export(nombre, CARPETA_SALIDA / "trabajo" / "inputs")
            if list(raiz.rglob("rag_run_manifest_v302.json")):
                RUTA_REFINADA = nombre
            elif list(raiz.rglob("run_manifest.json")) and list(raiz.rglob("lda_topic_terms.csv")):
                RUTA_INICIAL = nombre
    if not RUTA_INICIAL or not RUTA_REFINADA:
        raise ValueError("Falta identificar una etapa; completá las dos rutas.")
    datos = build_payload(RUTA_INICIAL, RUTA_REFINADA, CARPETA_SALIDA / "trabajo")
print("Etapas cargadas:", [(s["name"], s["export"]["filename"]) for s in datos["stages"]])

Seleccioná el ZIP de resultados v0.1/v0.2 o panorama_datos.json.


Saving Observatorio_NOA_TopicPanorama_Evolucion_v0_1_resultados.zip to Observatorio_NOA_TopicPanorama_Evolucion_v0_1_resultados.zip
Etapas cargadas: [('Inicial', 'outputs_1821.rar'), ('Refinada', 'observatorio_noa_rag_v302.zip')]


## 6. Revisar cobertura y un ejemplo de respaldo
Se muestra la cobertura de ambas etapas y el aporte documental al tópico T0 de La Rioja en la etapa refinada. Son cálculos descriptivos con los perfiles exportados; retirar un documento en el visor recalcula el promedio provincial.

In [11]:
import pandas as pd
from IPython.display import display, HTML, FileLink

display(pd.DataFrame([{
    "Etapa": s["name"], "Catalogados": len(s["documents"]),
    "Textos completos": sum(d["full"] for d in s["documents"]),
    "Provinciales con texto completo": sum(d["full"] and d["compare"] and not d["regional"] for d in s["documents"]),
    "Fragmentos": len(s["chunks"]), "Capa BERTopic": s["layers"]["bt"]["name"],
} for s in datos["stages"]]))
ejemplo = pd.DataFrame(influence(datos, "refined", "La Rioja", 0))
display(ejemplo[["doc_id", "source", "contribution_share", "mean_current", "mean_without_document", "delta_pp"]])
print("Las etiquetas interpretativas incluidas son propuestas pendientes de revisión.")

,Etapa,Catalogados,Textos completos,Provinciales con texto completo,Fragmentos,Capa BERTopic
0,Inicial,19,16,15,1704,BERTopic global · 3 tópicos
1,Refinada,19,19,18,1761,BERTopic sustantivo · 8 microtópicos


,doc_id,source,contribution_share,mean_current,mean_without_document,delta_pp
0,DOC_002,CFI,0.074765,0.186435,0.215621,2.918524
1,DOC_003,CFI,0.699221,0.186435,0.070095,-11.634072
2,DOC_017,SAIJ,0.015015,0.186435,0.229545,4.310964
3,DOC_018,SAIJ,0.101066,0.186435,0.209491,2.305601
4,DOC_019,SAIJ,0.109932,0.186435,0.207425,2.098984


Las etiquetas interpretativas incluidas son propuestas pendientes de revisión.


## 7. Leer las nuevas medidas

**Perfil observado:** media de los perfiles documentales, cada documento con igual peso. **Perfil equilibrado:** media de las medias CFI y SAIJ, con 50 % por fuente; queda no disponible si una fuente está ausente.

**Aporte al tópico:** $q_{dt}=w_d p_{dt}/\sum_d w_d p_{dt}$. Suma 100 % entre documentos activos. Difiere de la afinidad temática individual del documento. La exclusión temporal recalcula promedios y relaciones sobre los perfiles fijos. Si se pierde una fuente, la media equilibrada queda no disponible.

**Concentración relativa:** $L_{pt}=P(t\mid p)/\operatorname{media}_{p'}P(t\mid p')$, con el mismo peso por provincia disponible. Requiere al menos dos provincias. El filtro provincial conserva esa referencia comparativa; los demás filtros y exclusiones sí se aplican.

El soporte cuenta documentos donde un tópico está entre las dos mayores afinidades, incluidos empates. En K=2 todos los documentos cumplen el criterio. Los umbrales de la vista son exploratorios. Estas medidas no son probabilidades calibradas ni indicadores de presupuesto o impacto.

In [12]:
ruta_html, informe_validacion = write_project(datos, "visor_template.html", CARPETA_SALIDA)
print("Visor y tablas generados:", ruta_html.name)
print("Nuevas tablas:", informe_validacion["analytical_tables"])
display(FileLink(str(ruta_html)))
if MOSTRAR_VISOR:
    display(HTML(ruta_html.read_text(encoding="utf-8")))
else:
    print("HTML autónomo generado; visualización embebida desactivada.")

Output hidden; open in https://colab.research.google.com to view.

## 8. Guardar resultados y sesiones
El ZIP contiene el HTML, las auditorías, las tablas analíticas y las fuentes. Las tablas generales corresponden al escenario completo. Dentro del visor, **Descargar tabla actual** conserva los filtros y exclusiones de la vista. **Guardar sesión** registra escenarios, etiquetas, fundamentos y revisiones, y permite volver a cargarlos.

Las sesiones v0.1 son compatibles; sus etiquetas sin evidencia de validación se consideran propuestas. Las figuras SVG/PNG se generan desde la herramienta analítica seleccionada.

In [13]:
import zipfile
documentacion = '# Observatorio NOA · TopicPanorama · v0.2\n\n## Explorar directamente\n\nAbrí `Observatorio_NOA_TopicPanorama_Evolucion_v0_2.html` en un navegador. Contiene los datos y funciona sin conexión. Las nuevas herramientas están en **Lectura provincial y respaldo documental**, debajo de los panoramas.\n\n## Ejecutar en Colab\n\n1. Abrí `Observatorio_NOA_TopicPanorama_Evolucion_v0_2.ipynb` y ejecutá las celdas en orden.\n2. El modo predeterminado, `resultados`, solicita un único archivo: el ZIP de resultados de la v0.1 que ya ejecutaste, o un ZIP de resultados de la v0.2. También acepta `panorama_datos.json`.\n3. Para reconstruir desde las exportaciones originales, elegí `MODO_CARGA = "originales"` y cargá `outputs_1821.rar` y `observatorio_noa_rag_v302.zip`. Se admiten los nombres con sufijos y carpetas extraídas.\n4. El visor se genera en ambos modos. Se pueden descargar todas las tablas y las fuentes en un ZIP.\n\nLa ruta de resultados verifica las identidades y huellas de los textos conservados en el JSON; mantiene las comprobaciones de embeddings de la ejecución original, sin volver a cargar matrices ausentes del ZIP de resultados. La ruta original vuelve a verificar las matrices disponibles. No se vuelven a ajustar modelos.\n\n## Tres incorporaciones\n\n### Perfiles y fuentes\n\nEl panel muestra juntos el promedio observado, con igual peso por documento, y el promedio equilibrado, con 50 % para CFI y 50 % para SAIJ dentro de cada provincia. El equilibrio queda **no disponible** si falta una fuente. Una selección de fuente única utiliza la ponderación por documento.\n\n### Aporte y exclusión\n\nSeleccioná etapa, provincia y tópico. El aporte de cada documento al tópico suma 100 % entre documentos activos. La tabla distingue aporte, afinidad documental y perfil obtenido al retirar ese documento. El CSV incluye también el peso utilizado en el promedio.\n\nMarcar **Excluir** retira temporalmente el documento de ambas etapas, por su identidad documental. Actualiza perfiles, relaciones y concentraciones; mantiene la auditoría de los registros originales. **Restaurar** revierte el escenario. La comparación conserva los perfiles documentales calculados; no representa un nuevo ajuste LDA o BERTopic.\n\nSi al retirar un documento se pierde una fuente, la media equilibrada queda no disponible. La media observada puede seguir consultándose. Guardar sesión conserva el escenario y las revisiones.\n\n### Concentración relativa\n\nSe calcula `L(provincia,tópico) = afinidad provincial / promedio de afinidades de las provincias disponibles`, con el mismo peso por provincia. Un valor de 1 es el promedio; 1,5 expresa una afinidad 50 % mayor. Se requieren al menos dos provincias. El filtro provincial conserva la referencia, mientras los filtros de fuente, período, búsqueda, universo y exclusiones sí se aplican a ella.\n\nEl soporte cuenta documentos en los que el tópico está entre las dos mayores afinidades, incluidos empates. En K=2 todos los documentos cumplen el criterio. El borde resalta, de forma exploratoria, concentración ≥ 1,2 y soporte en al menos dos documentos; ambos umbrales se pueden modificar. Estas cifras no certifican relevancia política ni confianza estadística.\n\n## Etiquetas interpretativas\n\nEl modo inicial conserva las palabras originales. Se puede elegir **Propuestas interpretativas** o **Solo interpretaciones validadas**. Hay ocho propuestas para el nivel sustantivo refinado, acompañadas por pasajes concretos. Todas empiezan pendientes de validación.\n\nPara marcar una etiqueta como validada, el investigador debe registrar etiqueta, fundamento y referencia a un documento o fragmento exportado. Las etiquetas y notas se guardan como anotaciones, junto con fecha de revisión. Las sesiones v0.1 se pueden importar; las etiquetas anteriores se consideran propuestas si carecen de esos registros.\n\nLa discrepancia entre el título de DOC_016 —bosques nativos— y la referencia del texto a bosques cultivados se conserva como observación pendiente. El título original permanece intacto.\n\n## Archivos analíticos\n\n- `perfiles_observados_y_equilibrados.csv`: ambas ponderaciones y disponibilidad.\n- `aporte_documental_y_exclusion.csv`: aporte y exclusión individual para cada etapa, capa, provincia y tópico, en el escenario completo.\n- `concentracion_relativa.csv`: concentración, referencia y soporte.\n- `etiquetas_propuestas.csv`, `revision_catalogacion.csv` y `metodos_analiticos.json`.\n- Auditorías documentales y de fragmentos y correspondencias entre etapas conservadas.\n- `figuras/`: figuras de las nuevas herramientas exportadas desde el visor.\n\nLas tablas generales incluidas corresponden al escenario completo. **Descargar tabla actual** en el visor incluye los filtros y exclusiones en cada fila. SVG y PNG exportan la herramienta analítica actualmente seleccionada.\n\n## Alcance\n\nLas etapas identifican versiones de procesamiento; los períodos corresponden a años documentales. Los identificadores de tópicos son locales a cada ajuste. Inicial: tres tópicos BERTopic globales. Refinada: ocho microtópicos sustantivos, con un modelo global de dos tópicos conservado en el manifiesto. Más tópicos no implica mayor calidad.\n\nLa etapa inicial contiene 16 textos completos; la refinada, 19. Los 1.704 fragmentos originales se conservan con igual identificador y texto; se agregan 57 por OCR. ATACALAR se consulta como contexto regional y queda fuera de los perfiles provinciales.\n\nLos perfiles iniciales BERTopic se condicionan a la masa de los tres tópicos, excluyendo la masa atípica. Los ocho perfiles sustantivos refinados ya vienen normalizados. Afinidades y similitudes no son probabilidades calibradas. Las palabras ponderadas LDA se conservan donde fueron exportadas; los descriptores refinados carecen de sus pesos individuales.\n\nEsta es una adaptación de la exploración de TopicPanorama con relaciones documentales, evidencia y anotaciones. No replica CTM, BRT ni el optimizador húngaro incremental original. Dos estados permiten documentar cambios conjuntos; no atribuir causalmente diferencias a cada intervención de limpieza o modelado.\n\nReferencia: Liu, S., Wang, X., Chen, J., Zhu, J. y Guo, B. (2014). TopicPanorama: a Full Picture of Relevant Topics. IEEE VAST. https://doi.org/10.1109/VAST.2014.7042494\n'
(CARPETA_SALIDA / "LEEME.md").write_text(documentacion, encoding="utf-8")
ruta_zip = CARPETA_SALIDA / "Observatorio_NOA_TopicPanorama_Evolucion_v0_2_resultados.zip"
with zipfile.ZipFile(ruta_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CARPETA_SALIDA.iterdir()):
        if p.is_file() and p != ruta_zip and p.suffix != ".zip":
            z.write(p, p.name)
    for nombre in ["noa_analysis.py", "noa_panorama.py", "visor_template.html"]:
        z.write(nombre, "fuentes/" + nombre)
display(FileLink(str(ruta_zip)))
if DESCARGAR_ZIP_EN_COLAB:
    try:
        from google.colab import files
        files.download(str(ruta_zip))
    except ImportError:
        print("Fuera de Colab: abrí la carpeta o usá el enlace al ZIP.")

/content/observatorio_noa_panorama_v02_salida/Observatorio_NOA_TopicPanorama_Evolucion_v0_2_resultados.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Referencia y alcance
Liu, S., Wang, X., Chen, J., Zhu, J. y Guo, B. (2014). *TopicPanorama: a Full Picture of Relevant Topics*. IEEE VAST. https://doi.org/10.1109/VAST.2014.7042494

La cartografía adapta navegación, selección enlazada, evidencia, comparación y revisión. Las etapas preservadas tienen alcances diferentes: tres tópicos BERTopic globales al inicio y ocho microtópicos sustantivos en la vista refinada. Se mantienen también sus capas LDA exportadas.

Los indicadores incorporados son descriptivos. Las etiquetas propuestas requieren validación contextual. La reoptimización húngara incremental del sistema original no está implementada; las revisiones son anotaciones reversibles. Dos estados permiten documentar cambios conjuntos y no atribuir por separado efectos causales al OCR, limpieza o modelado.